# Physics-Informed GNN for Model 2 1D with Static Features, Graph Attention, Multi-Step Rollout, and RainGate

This notebook trains a physics-informed graph neural network for `Model 2 / 1D` water-level forecasting in the Kaggle environment.
It uses node static features, optional dynamic extra features, graph-based message passing, multi-step rollout training, and a rain-conditioned `RainGate` that modulates hidden states from rainfall context.

## What this notebook does
- predicts `water_level` for `Model 2 / 1D`
- uses a graph neural network over the 1D node graph
- supports both mean aggregation and graph-attention aggregation
- uses multi-step rollout loss during training when enabled
- applies `RainGate` to condition the model on rainfall history and event-level rain context
- writes submission and bundle outputs for downstream ensemble notebooks

## Inputs used by the model
- lagged `water_level`
- lagged rainfall features
- optional dynamic extra columns
- node static features
- learned node embeddings

## Training objective
The base objective is one-step delta prediction with the existing physics-informed regularizers.
When rollout loss is enabled, the model is also trained to stay consistent across several recursive future steps.

## Output artifacts
- prediction table for test events
- Kaggle submission file
- bundle directory containing OOF predictions, test predictions, manifest, and CV summary


In [1]:
#!/usr/bin/env python3
# Physics-Informed GNN (Model2 1D water_level) + RainGate + static features + optional AUX targets + Bundle saving
# Kaggle-ready version (works on Kaggle Notebook environment)
#
# Notes:
# - Uses /kaggle/input for dataset & competition sample_submission by default
# - Writes outputs to /kaggle/working (Kaggle output area)
# - Still supports overriding paths via env vars:
#     DATASET_ROOT, SAMPLE_SUBMISSION_PATH, OUTPUT_DIR
#
# Typical Kaggle setup:
# - Add dataset: "mtmrs1/urban-flood-bench-re" (or equivalent) to Notebook "Data"
# - Ensure competition dataset "urban-flood-modelling" is available (sample_submission.csv)
#
# Environment overrides:
#   DATASET_ROOT=/kaggle/input/<your-dataset-folder>
#   SAMPLE_SUBMISSION_PATH=/kaggle/input/competitions/urban-flood-modelling/sample_submission.csv
#   OUTPUT_DIR=/kaggle/working/run_custom
#
# This script is intentionally self-contained.

import os
import gc
import time
import copy
import json
import warnings
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")


# =========================
# Config
# =========================

def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, str(default)))


def env_float(name: str, default: float) -> float:
    return float(os.environ.get(name, str(default)))


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


def parse_str_list(text: str):
    return [c.strip() for c in text.split(",") if c.strip()]


SEED = env_int("SEED", 2026)

# --- Kaggle defaults (override if needed) ---
DEFAULT_DATASET_ROOT = "/kaggle/input/datasets/mtmrs1/urban-flood-bench-re"
DEFAULT_SAMPLE_SUBMISSION = "/kaggle/input/competitions/urban-flood-modelling/sample_submission.csv"

DATASET_ROOT = Path(os.environ.get("DATASET_ROOT", DEFAULT_DATASET_ROOT))
SAMPLE_SUBMISSION_PATH = os.environ.get("SAMPLE_SUBMISSION_PATH", DEFAULT_SAMPLE_SUBMISSION)

# --- fixed target ---
TARGET = (2, 1)  # model2, 1d
TARGET_COL = "water_level"
TARGET_MODEL_ID, TARGET_NODE_TYPE = TARGET

# folds / sampling
N_FOLDS = env_int("N_FOLDS", 5)
RUN_SINGLE_FOLD = env_bool("RUN_SINGLE_FOLD", False)
FOLD_INDEX = env_int("FOLD_INDEX", 0)
EVENT_SPLIT_SEED = env_int("EVENT_SPLIT_SEED", 2026)

START_T = env_int("START_T", 10)
Y_LAGS = env_int("Y_LAGS", 10)
RAIN_LAGS = env_int("RAIN_LAGS", 80)

# rain3 anchors (legacy path kept but disabled by default)
RAIN3_ENABLED = env_bool("RAIN3_ENABLED", False)
RAIN_ANCHOR_QLOW = env_float("RAIN_ANCHOR_QLOW", 0.10)
RAIN_ANCHOR_QMID = env_float("RAIN_ANCHOR_QMID", 0.50)
RAIN_ANCHOR_QHIGH = env_float("RAIN_ANCHOR_QHIGH", 0.90)
RAIN_ANCHOR_BW_SCALE = env_float("RAIN_ANCHOR_BW_SCALE", 0.75)
RAIN_ANCHOR_MIN_BW = env_float("RAIN_ANCHOR_MIN_BW", 1e-4)

# extra + aux
EXTRA_COLS_REQ = parse_str_list(os.environ.get("EXTRA_COLS", ""))  # feature-only extras
JOINT_AUX_COLS_REQ = parse_str_list(os.environ.get("JOINT_AUX_COLS", "inlet_flow"))  # extra cols to predict delta for
EXTRA_LAGS = env_int("EXTRA_LAGS", 10)
AUX_LOSS_WEIGHT = env_float("AUX_LOSS_WEIGHT", 1.0)

# limits
MAX_TRAIN_EVENTS = env_int("MAX_TRAIN_EVENTS", 0)
MAX_TEST_EVENTS = env_int("MAX_TEST_EVENTS", 0)
MAX_TRAIN_ROWS = env_int("MAX_TRAIN_ROWS", 0)
MAX_VAL_ROWS = env_int("MAX_VAL_ROWS", 0)

# train
EPOCHS = env_int("EPOCHS", 200)
LR = env_float("LR", 3e-4)
WEIGHT_DECAY = env_float("WEIGHT_DECAY", 3e-6)
PATIENCE = env_int("PATIENCE", 25)
MIN_LR = env_float("MIN_LR", 2e-5)
LR_FACTOR = env_float("LR_FACTOR", 0.5)
LR_PATIENCE = env_int("LR_PATIENCE", 4)
GRAD_CLIP_NORM = env_float("GRAD_CLIP_NORM", 1.0)

EPOCH_OOF_EVAL_EVERY = env_int("EPOCH_OOF_EVAL_EVERY", 5)
EPOCH_OOF_MAX_EVENTS = env_int("EPOCH_OOF_MAX_EVENTS", 0)
OOF_LB_EVAL = env_bool("OOF_LB_EVAL", True)
OOF_LB_MAX_EVENTS = env_int("OOF_LB_MAX_EVENTS", 0)

# model
GNN_D_MODEL = env_int("GNN_D_MODEL", 128)
GNN_N_LAYERS = env_int("GNN_N_LAYERS", 12)
GNN_DROPOUT = env_float("GNN_DROPOUT", 0.0)
NODE_EMBED_DIM = env_int("NODE_EMBED_DIM", 64)
GRAPH_BATCH_SIZE = env_int("GRAPH_BATCH_SIZE", 64)
GNN_LAYER_TYPE = os.environ.get("GNN_LAYER_TYPE", "attn").strip().lower()
ATTN_DROPOUT = env_float("ATTN_DROPOUT", 0.0)
ATTN_USE_OUT_PROJ = env_bool("ATTN_USE_OUT_PROJ", True)
RAIN_GATE_ENABLE = env_bool("RAIN_GATE_ENABLE", True)
RAIN_GATE_HIDDEN = env_int("RAIN_GATE_HIDDEN", 64)
RAIN_GATE_SCALE = env_float("RAIN_GATE_SCALE", 0.5)
ROLLOUT_LOSS_ENABLE = env_bool("ROLLOUT_LOSS_ENABLE", True)
ROLLOUT_STEPS = env_int("ROLLOUT_STEPS", 2)
ROLLOUT_WEIGHT = env_float("ROLLOUT_WEIGHT", 0.2)
ROLLOUT_BATCH_SIZE = env_int("ROLLOUT_BATCH_SIZE", 64)
ROLLOUT_MAX_SAMPLES = env_int("ROLLOUT_MAX_SAMPLES", 0)

LAMBDA_SMOOTH = env_float("LAMBDA_SMOOTH", 0.05)
LAMBDA_MASS = env_float("LAMBDA_MASS", 0.02)
LAMBDA_DYN = env_float("LAMBDA_DYN", 0.02)

PRED_BATCH_SIZE = env_int("PRED_BATCH_SIZE", 8192)
LOG_EVERY_EVENTS = env_int("LOG_EVERY_EVENTS", 1)
ALLOW_PARTIAL_TARGETS = env_bool("ALLOW_PARTIAL_TARGETS", True)

# safety clips
SAFE_FEATURE_CLIP = 1e6
SAFE_LEVEL_CLIP = 1e6
SAFE_DELTA_CLIP = 1e4

KEY_COLS = ["model_id", "event_id", "node_type", "node_id", "timestep"]

STD_DEV = {
    (1, 1): 16.877747,
    (1, 2): 14.378797,
    (2, 1): 3.191784,
    (2, 2): 2.727131,
}

USE_AMP_ENABLED = env_bool("USE_AMP", True)
DEVICE = "cpu"
USE_AMP = False
_CUDA_RUNTIME_ENABLED = False
CPU_COUNT = max(1, int(os.cpu_count() or 1))
NUM_WORKERS = env_int("NUM_WORKERS", 2 if CPU_COUNT >= 4 else 0)
PARALLEL_FOLDS_ENABLE = env_bool("PARALLEL_FOLDS_ENABLE", True)
_PARALLEL_GPU_IDS_RAW = parse_str_list(os.environ.get("PARALLEL_GPU_IDS", "0,1"))
PARALLEL_GPU_IDS = [int(x) for x in _PARALLEL_GPU_IDS_RAW] if _PARALLEL_GPU_IDS_RAW else []
PARALLEL_MAX_WORKERS = env_int("PARALLEL_MAX_WORKERS", 0)
PARALLEL_START_METHOD = os.environ.get("PARALLEL_START_METHOD", "fork").strip().lower()

# --- outputs: Kaggle output area ---
DEFAULT_OUTPUT_DIR = Path("/kaggle/working") / f"run_{time.strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = Path(os.environ.get("OUTPUT_DIR", str(DEFAULT_OUTPUT_DIR)))
RUN_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = str(RUN_DIR / "submission.csv")
PRED_TABLE_PATH = str(RUN_DIR / "pred_physgnn_m2_1d_static_attn_rollout_v2_raingate.csv")
STATIC_FEATURES_ENABLE = env_bool("STATIC_FEATURES_ENABLE", True)
STATIC_COLS_REQ = parse_str_list(os.environ.get("STATIC_COLS", "position_x,position_y,depth,invert_elevation,surface_elevation,base_area"))
STATIC_STANDARDIZE = env_bool("STATIC_STANDARDIZE", False)

BUNDLE_DIR = RUN_DIR / "bundle_physics_informed_gnn_m2_1d_static_attn_rollout_v2_raingate"
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_NAME = BUNDLE_DIR.name


# =========================
# Checks
# =========================

if N_FOLDS < 2:
    raise RuntimeError(f"N_FOLDS must be >=2, got {N_FOLDS}")
if START_T < 1:
    raise RuntimeError(f"START_T must be >=1, got {START_T}")
if Y_LAGS < 1:
    raise RuntimeError(f"Y_LAGS must be >=1, got {Y_LAGS}")
if RAIN_LAGS < 0:
    raise RuntimeError(f"RAIN_LAGS must be >=0, got {RAIN_LAGS}")
if START_T < Y_LAGS:
    raise RuntimeError(f"START_T must be >= Y_LAGS: START_T={START_T}, Y_LAGS={Y_LAGS}")
if GRAPH_BATCH_SIZE < 1:
    raise RuntimeError(f"GRAPH_BATCH_SIZE must be >=1, got {GRAPH_BATCH_SIZE}")
if GNN_LAYER_TYPE not in {"mean", "attn"}:
    raise RuntimeError(f"GNN_LAYER_TYPE must be one of {{'mean', 'attn'}}, got {GNN_LAYER_TYPE}")
if ROLLOUT_STEPS < 1:
    raise RuntimeError(f"ROLLOUT_STEPS must be >=1, got {ROLLOUT_STEPS}")
if ROLLOUT_BATCH_SIZE < 1:
    raise RuntimeError(f"ROLLOUT_BATCH_SIZE must be >=1, got {ROLLOUT_BATCH_SIZE}")
if PARALLEL_START_METHOD not in {"fork", "spawn", "forkserver"}:
    raise RuntimeError(f"PARALLEL_START_METHOD must be one of {{'fork', 'spawn', 'forkserver'}}, got {PARALLEL_START_METHOD}")
if RAIN3_ENABLED:
    if not (0.0 <= RAIN_ANCHOR_QLOW < RAIN_ANCHOR_QMID < RAIN_ANCHOR_QHIGH <= 1.0):
        raise RuntimeError(
            "RAIN_ANCHOR quantiles must satisfy 0<=QLOW<QMID<QHIGH<=1, got "
            f"{RAIN_ANCHOR_QLOW}, {RAIN_ANCHOR_QMID}, {RAIN_ANCHOR_QHIGH}"
        )
    if RAIN_ANCHOR_BW_SCALE <= 0.0:
        raise RuntimeError(f"RAIN_ANCHOR_BW_SCALE must be >0, got {RAIN_ANCHOR_BW_SCALE}")
    if RAIN_ANCHOR_MIN_BW <= 0.0:
        raise RuntimeError(f"RAIN_ANCHOR_MIN_BW must be >0, got {RAIN_ANCHOR_MIN_BW}")


# =========================
# Utils
# =========================

def log(msg: str) -> None:
    ts = time.strftime("%H:%M:%S")
    print(f"[{ts}] {msg}", flush=True)


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if _CUDA_RUNTIME_ENABLED:
        torch.cuda.manual_seed_all(seed)


def detect_gpu_count_no_torch() -> int:
    visible = os.environ.get("CUDA_VISIBLE_DEVICES", "").strip()
    if visible:
        toks = [x.strip() for x in visible.split(",") if x.strip() and x.strip() != "-1"]
        return len(toks)
    try:
        proc = subprocess.run(
            ["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
            check=True,
            capture_output=True,
            text=True,
        )
        return len([ln for ln in proc.stdout.splitlines() if ln.strip()])
    except Exception:
        return 0


def set_runtime_device(gpu_id: int | None = None) -> None:
    global DEVICE, USE_AMP, _CUDA_RUNTIME_ENABLED
    if gpu_id is not None:
        torch.cuda.set_device(gpu_id)
        DEVICE = "cuda"
        _CUDA_RUNTIME_ENABLED = True
    else:
        DEVICE = "cpu"
        _CUDA_RUNTIME_ENABLED = False
    USE_AMP = USE_AMP_ENABLED and DEVICE == "cuda"


def state_dict_to_cpu(state_dict):
    return {k: v.detach().cpu() for k, v in state_dict.items()}


def save_table_auto(df: pd.DataFrame, stem_path: Path) -> Path:
    stem_path.parent.mkdir(parents=True, exist_ok=True)
    parquet_path = stem_path.with_suffix(".parquet")
    try:
        df.to_parquet(parquet_path, index=False)
        return parquet_path
    except Exception as exc:
        csv_path = stem_path.with_suffix(".csv")
        df.to_csv(csv_path, index=False)
        log(f"parquet save failed ({parquet_path.name}); fallback csv ({csv_path.name}) reason={exc}")
        return csv_path


# =========================
# Paths / I/O
# =========================

def resolve_models_root(dataset_root: Path) -> Path:
    # Accept a Kaggle dataset mounted at /kaggle/input/<dataset_name>
    # where data could be in <root>/Models or directly under <root>
    for p in [dataset_root / "Models", dataset_root]:
        if (p / "Model_2" / "train").exists() and (p / "Model_2" / "test").exists():
            return p

    # Helpful fallback: if user passed /kaggle/input and dataset is nested one level.
    if dataset_root.exists() and dataset_root.is_dir():
        for child in dataset_root.iterdir():
            if not child.is_dir():
                continue
            for p in [child / "Models", child]:
                if (p / "Model_2" / "train").exists() and (p / "Model_2" / "test").exists():
                    log(f"resolve_models_root: auto-detected under {child}")
                    return p

    raise RuntimeError(f"models root not found under DATASET_ROOT={dataset_root}")


def resolve_sample_submission(explicit: str, models_root: Path = None) -> Path:
    cands = [Path(explicit)]
    # common Kaggle competition location
    cands.append(Path("/kaggle/input/competitions/urban-flood-modelling/sample_submission.csv"))
    cands.append(Path("/kaggle/input/competitions/urban-flood-modeling/sample_submission.csv"))
    if models_root is not None:
        cands.append(Path(models_root) / "sample_submission.csv")
        cands.append(Path(models_root).parent / "sample_submission.csv")

    for p in cands:
        if p.exists():
            return p

    tried = [str(p) for p in cands]
    raise RuntimeError(f"sample_submission.csv not found. tried={tried}")


def model_dir(models_root: Path, model_id: int) -> Path:
    p = models_root / f"Model_{model_id}"
    if not (p / "train").exists() or not (p / "test").exists():
        raise RuntimeError(f"invalid model dir: {p}")
    return p


def node_static_name(node_type: int) -> str:
    return "1d_nodes_static.csv" if node_type == 1 else "2d_nodes_static.csv"


def node_dynamic_name(node_type: int) -> str:
    return "1d_nodes_dynamic_all.csv" if node_type == 1 else "2d_nodes_dynamic_all.csv"


def list_event_ids(mdir: Path, split: str):
    out = []
    root = mdir / split
    for d in root.iterdir():
        if d.is_dir() and d.name.startswith("event_"):
            try:
                out.append(int(d.name.split("_")[-1]))
            except ValueError:
                pass
    return sorted(out)


def load_node_ids(mdir: Path, node_type: int):
    df = pd.read_csv(mdir / "train" / node_static_name(node_type), usecols=["node_idx"])
    return np.sort(df["node_idx"].astype(np.int32).unique())


def load_1d_edge_index(mdir: Path, node_ids: np.ndarray):
    cands = [
        mdir / "train" / "1d_edge_index.csv",
        mdir / "test" / "1d_edge_index.csv",
        mdir / "train" / "1d_edges_index.csv",
        mdir / "test" / "1d_edges_index.csv",
    ]
    edge_path = None
    for p in cands:
        if p.exists():
            edge_path = p
            break
    if edge_path is None:
        raise RuntimeError("1d_edge_index.csv not found in train/test")

    df = pd.read_csv(edge_path)
    cols = {c.lower(): c for c in df.columns}
    from_col = None
    to_col = None
    for key in ("from_node", "src_node", "source_node", "u", "from"):
        if key in cols:
            from_col = cols[key]
            break
    for key in ("to_node", "dst_node", "target_node", "v", "to"):
        if key in cols:
            to_col = cols[key]
            break
    if from_col is None or to_col is None:
        raise RuntimeError(f"edge index columns not found in {edge_path}; cols={list(df.columns)}")

    id2pos = {int(nid): i for i, nid in enumerate(node_ids.tolist())}
    src = df[from_col].astype(np.int32).to_numpy()
    dst = df[to_col].astype(np.int32).to_numpy()

    edges = []
    for s, d in zip(src.tolist(), dst.tolist()):
        if s in id2pos and d in id2pos:
            u = id2pos[s]
            v = id2pos[d]
            edges.append((u, v))
            edges.append((v, u))

    if not edges:
        raise RuntimeError("no valid 1D edges after mapping edge_index to node_ids")

    e = np.asarray(edges, dtype=np.int64)
    return np.stack([e[:, 0], e[:, 1]], axis=0)


def detect_rain_col(mdir: Path, split: str, event_id: int):
    evdir = mdir / split / f"event_{event_id}" / "2d_nodes_dynamic_all.csv"
    cols = pd.read_csv(evdir, nrows=1).columns.tolist()
    for c in ("rainfall", "rainfall_depth", "rainfall_intensity"):
        if c in cols:
            return c
    raise RuntimeError("rainfall column not found")


def resolve_extra_cols(mdir: Path, split: str, event_id: int, node_type: int, requested_cols):
    if not requested_cols:
        return []
    dyn_path = mdir / split / f"event_{event_id}" / node_dynamic_name(node_type)
    cols = pd.read_csv(dyn_path, nrows=0).columns.tolist()
    active = [c for c in requested_cols if c in cols and c != TARGET_COL]
    missing = [c for c in requested_cols if c not in cols]
    if missing:
        log(f"extra columns missing and skipped: {missing}")
    return active


def load_static_node_features(
    mdir: Path,
    node_type: int,
    node_ids: np.ndarray,
    requested_cols,
    standardize: bool,
):
    static_path = mdir / "train" / node_static_name(node_type)
    full = pd.read_csv(static_path)
    if "node_idx" not in full.columns:
        raise RuntimeError(f"node_idx missing in static file: {static_path}")
    work = full.set_index("node_idx").reindex(node_ids)
    numeric_cols = []
    for col in work.columns.tolist():
        if col == "node_idx":
            continue
        if pd.api.types.is_numeric_dtype(work[col]):
            numeric_cols.append(col)
    if requested_cols:
        active_cols = [c for c in requested_cols if c in numeric_cols]
        missing = [c for c in requested_cols if c not in numeric_cols]
        if missing:
            log(f"static columns missing and skipped: {missing}")
    else:
        active_cols = numeric_cols
    if not STATIC_FEATURES_ENABLE or not active_cols:
        return np.empty((len(node_ids), 0), dtype=np.float32), active_cols
    feat = work[active_cols].to_numpy(dtype=np.float32, copy=False)
    col_mean = np.nanmean(feat, axis=0).astype(np.float32, copy=False)
    col_mean = np.where(np.isfinite(col_mean), col_mean, 0.0).astype(np.float32, copy=False)
    mask = ~np.isfinite(feat)
    if mask.any():
        feat = np.where(mask, col_mean[None, :], feat).astype(np.float32, copy=False)
    if standardize and feat.shape[1] > 0:
        mu = feat.mean(axis=0, dtype=np.float64).astype(np.float32)
        sigma = feat.std(axis=0, dtype=np.float64).astype(np.float32)
        sigma = np.where(sigma < 1e-6, 1.0, sigma).astype(np.float32)
        feat = ((feat - mu[None, :]) / sigma[None, :]).astype(np.float32, copy=False)
    np.nan_to_num(feat, copy=False, nan=0.0, posinf=SAFE_FEATURE_CLIP, neginf=-SAFE_FEATURE_CLIP)
    np.clip(feat, -SAFE_FEATURE_CLIP, SAFE_FEATURE_CLIP, out=feat)
    return feat.astype(np.float32, copy=False), active_cols


# =========================
# Rain helpers
# =========================

def build_rain_lags(rain: np.ndarray, rain_lags: int) -> np.ndarray:
    n = len(rain)
    out = np.zeros((n, rain_lags + 1), dtype=np.float32)
    if n == 0:
        return out
    out[:, 0] = rain.astype(np.float32, copy=False)
    for k in range(1, rain_lags + 1):
        if k >= n:
            break
        out[k:, k] = rain[: n - k]
    return out


def compute_rain_event_score(ev):
    rain = np.asarray(ev["rain"], dtype=np.float64)
    rain = np.nan_to_num(rain, nan=0.0, posinf=0.0, neginf=0.0)
    total_rain = float(np.sum(np.clip(rain, 0.0, None)))
    return float(np.log1p(total_rain))


def build_rain_anchor_params(train_events):
    event_ids = np.asarray([int(ev["event_id"]) for ev in train_events], dtype=np.int32)
    scores = np.asarray([compute_rain_event_score(ev) for ev in train_events], dtype=np.float64)

    q = np.quantile(scores, [RAIN_ANCHOR_QLOW, RAIN_ANCHOR_QMID, RAIN_ANCHOR_QHIGH]).astype(np.float64)
    centers = q.astype(np.float32, copy=False)

    spread = float(max(q[2] - q[0], RAIN_ANCHOR_MIN_BW))
    bandwidth = float(max(RAIN_ANCHOR_BW_SCALE * spread, RAIN_ANCHOR_MIN_BW))

    score_by_event = {int(eid): float(sc) for eid, sc in zip(event_ids.tolist(), scores.tolist())}
    return {"centers": centers, "bandwidth": bandwidth, "score_by_event": score_by_event}


def anchor_raw_weights(score: float, centers: np.ndarray, bandwidth: float):
    d = np.abs(np.asarray(score, dtype=np.float64) - centers.astype(np.float64))
    raw = np.maximum(0.0, 1.0 - d / float(bandwidth)).astype(np.float32, copy=False)
    if float(np.sum(raw)) <= 0.0:
        raw[:] = 0.0
        raw[int(np.argmin(d))] = 1.0
    return raw


def anchor_norm_weights(score: float, centers: np.ndarray, bandwidth: float):
    raw = anchor_raw_weights(score, centers, bandwidth)
    s = float(np.sum(raw))
    if s <= 0.0:
        return np.array([1.0, 0.0, 0.0], dtype=np.float32)
    return (raw / s).astype(np.float32, copy=False)


def build_anchor_graph_weights(graph_event_ids: np.ndarray, score_by_event: dict, centers: np.ndarray, bandwidth: float):
    graph_event_ids = np.asarray(graph_event_ids, dtype=np.int32)
    uniq_ids, inv = np.unique(graph_event_ids, return_inverse=True)

    uniq_raw = np.empty((uniq_ids.size, centers.size), dtype=np.float32)
    for i, eid in enumerate(uniq_ids.tolist()):
        score = float(score_by_event.get(int(eid), 0.0))
        uniq_raw[i, :] = anchor_raw_weights(score, centers, bandwidth)
    return uniq_raw[inv, :].astype(np.float32, copy=False)


# =========================
# Data loading
# =========================

def load_event(
    mdir: Path,
    split: str,
    event_id: int,
    node_type: int,
    node_ids: np.ndarray,
    target_col: str,
    rain_col: str,
    extra_cols_active: list,
):
    evdir = mdir / split / f"event_{event_id}"

    dyn_path = evdir / node_dynamic_name(node_type)
    cols = pd.read_csv(dyn_path, nrows=0).columns.tolist()
    extra_cols_present = [c for c in extra_cols_active if c in cols and c != target_col]

    usecols = ["timestep", "node_idx", target_col] + extra_cols_present
    df = pd.read_csv(dyn_path, usecols=usecols)

    p = (
        df.pivot(index="timestep", columns="node_idx", values=target_col)
        .sort_index()
        .reindex(columns=node_ids)
    )

    r = pd.read_csv(evdir / "2d_nodes_dynamic_all.csv", usecols=["timestep", rain_col])
    rain = r.drop_duplicates("timestep").sort_values("timestep")[rain_col].to_numpy(dtype=np.float32)

    t = p.index.to_numpy(dtype=np.int32)
    y = p.to_numpy(dtype=np.float32)

    extra = {}
    for col in extra_cols_active:
        if col in extra_cols_present:
            px = (
                df.pivot(index="timestep", columns="node_idx", values=col)
                .sort_index()
                .reindex(columns=node_ids)
                .reindex(index=p.index)
            )
            extra[col] = px.to_numpy(dtype=np.float32)
        else:
            extra[col] = np.full_like(y, np.nan, dtype=np.float32)

    n = min(len(t), len(rain), y.shape[0])
    return {
        "event_id": int(event_id),
        "t": t[:n],
        "y": y[:n],
        "rain": rain[:n],
        "extra": {k: v[:n] for k, v in extra.items()},
    }


def split_events_kfold(events, n_folds: int, seed: int):
    arr = np.asarray(sorted(events), dtype=np.int32)
    rng = np.random.RandomState(seed)
    rng.shuffle(arr)
    folds = np.array_split(arr, n_folds)
    return [sorted(x.tolist()) for x in folds]


def fold_train_events(folds, holdout_idx: int):
    out = []
    for i, f in enumerate(folds):
        if i == holdout_idx:
            continue
        out.extend(f)
    return sorted(out)


def compute_node_means_from_events(events):
    n_nodes = events[0]["y"].shape[1]
    sums = np.zeros((n_nodes,), dtype=np.float64)
    cnt = 0
    for ev in events:
        arr = ev["y"]
        sums += np.nan_to_num(arr, nan=0.0).sum(axis=0)
        cnt += arr.shape[0]
    return (sums / max(cnt, 1)).astype(np.float32)


def compute_extra_means_from_events(events, extra_cols):
    out = {}
    for col in extra_cols:
        n_nodes = events[0]["y"].shape[1]
        sums = np.zeros((n_nodes,), dtype=np.float64)
        cnt = 0
        for ev in events:
            arr = ev["extra"][col]
            sums += np.nan_to_num(arr, nan=0.0).sum(axis=0)
            cnt += arr.shape[0]
        out[col] = (sums / max(cnt, 1)).astype(np.float32)
    return out


def fill_missing(events, node_means, extra_means=None):
    for ev in events:
        arr = ev["y"]
        mask = ~np.isfinite(arr)
        if mask.any():
            ev["y"] = np.where(mask, node_means[np.newaxis, :], arr).astype(np.float32, copy=False)
        if extra_means:
            for col, means in extra_means.items():
                ex = ev["extra"][col]
                ex_mask = ~np.isfinite(ex)
                if ex_mask.any():
                    ev["extra"][col] = np.where(ex_mask, means[np.newaxis, :], ex).astype(np.float32, copy=False)


def choose_time_indices(events, start_t: int, max_rows: int, n_nodes: int, seed: int):
    rows_possible = 0
    for ev in events:
        rows_possible += max(0, len(ev["t"]) - start_t) * n_nodes

    if max_rows <= 0 or rows_possible <= max_rows:
        return [None] * len(events), rows_possible, rows_possible

    rng = np.random.RandomState(seed)
    picks = []
    rows_kept = 0

    for ev in events:
        n = len(ev["t"])
        if n <= start_t:
            picks.append(np.array([], dtype=np.int32))
            continue

        rows_ev = (n - start_t) * n_nodes
        keep_t = max(1, int(round((rows_ev / rows_possible) * max_rows)))
        keep_t = min(keep_t, n - start_t)

        idx = rng.choice(n - start_t, size=keep_t, replace=False).astype(np.int32)
        picks.append(idx)
        rows_kept += keep_t * n_nodes

    return picks, rows_possible, rows_kept


def build_samples(
    events,
    start_t: int,
    y_lags: int,
    rain_lags: int,
    extra_cols,
    joint_aux_cols,
    extra_lags: int,
    static_feat: np.ndarray | None,
    max_rows: int,
    seed: int,
    return_graph_event_ids: bool = False,
):
    n_nodes = events[0]["y"].shape[1]
    node_pos = np.arange(n_nodes, dtype=np.float32) / max(1.0, float(n_nodes - 1))

    picks, rows_possible, rows_kept = choose_time_indices(events, start_t, max_rows, n_nodes, seed)

    extra_dim = len(extra_cols) * max(0, extra_lags)
    static_dim = 0 if static_feat is None else int(static_feat.shape[1])
    feat_dim = 1 + static_dim + y_lags + extra_dim + (rain_lags + 1) + 1

    x = np.empty((rows_kept, feat_dim), dtype=np.float32)
    y = np.empty((rows_kept,), dtype=np.float32)

    n_aux = len(joint_aux_cols)
    y_aux = (
        np.empty((rows_kept, n_aux), dtype=np.float32)
        if n_aux > 0
        else np.empty((rows_kept, 0), dtype=np.float32)
    )

    graph_event_ids = [] if return_graph_event_ids else None

    ptr = 0
    total = len(events)

    for j, ev in enumerate(events, 1):
        arr = ev["y"]
        rain = ev["rain"]
        extra = ev["extra"]

        n = min(arr.shape[0], len(rain))
        if n <= start_t:
            continue

        arr = arr[:n]
        rain = rain[:n]
        extra = {k: v[:n] for k, v in extra.items()}

        rain_hist = build_rain_lags(rain, rain_lags)
        rain_mean = float(np.mean(rain))

        rel = picks[j - 1]
        if rel is None:
            rel = np.arange(n - start_t, dtype=np.int32)
        if rel.size == 0:
            continue

        for r in rel.tolist():
            tt = start_t + int(r)

            step_y = np.stack([arr[tt - 1 - k, :] for k in range(y_lags)], axis=1).astype(
                np.float32, copy=False
            )

            step_extra = []
            if extra_lags > 0:
                for col in extra_cols:
                    ex = extra[col]
                    ex_lag = np.stack([ex[tt - 1 - k, :] for k in range(extra_lags)], axis=1).astype(
                        np.float32, copy=False
                    )
                    step_extra.append(ex_lag)

            step_r = np.tile(rain_hist[tt, : rain_lags + 1][None, :], (n_nodes, 1)).astype(np.float32, copy=False)
            step_mean = np.full((n_nodes, 1), rain_mean, dtype=np.float32)

            feat_parts = [node_pos[:, None]]
            if static_feat is not None and static_feat.shape[1] > 0:
                feat_parts.append(static_feat)
            feat_parts.append(step_y)
            if step_extra:
                feat_parts.append(np.concatenate(step_extra, axis=1))
            feat_parts.extend([step_r, step_mean])

            feat = np.concatenate(feat_parts, axis=1)
            target = (arr[tt, :] - arr[tt - 1, :]).astype(np.float32, copy=False)

            if n_aux > 0:
                target_aux = np.stack(
                    [(extra[col][tt, :] - extra[col][tt - 1, :]).astype(np.float32, copy=False) for col in joint_aux_cols],
                    axis=1,
                ).astype(np.float32, copy=False)

            x[ptr : ptr + n_nodes, :] = feat
            y[ptr : ptr + n_nodes] = target
            if n_aux > 0:
                y_aux[ptr : ptr + n_nodes, :] = target_aux

            ptr += n_nodes
            if return_graph_event_ids:
                graph_event_ids.append(int(ev["event_id"]))

        if j % max(1, LOG_EVERY_EVENTS) == 0 or j == total:
            log(f"sample build progress: {j}/{total} events")

    x = x[:ptr]
    y = y[:ptr]
    y_aux = y_aux[:ptr]

    np.nan_to_num(x, copy=False, nan=0.0, posinf=SAFE_FEATURE_CLIP, neginf=-SAFE_FEATURE_CLIP)
    np.clip(x, -SAFE_FEATURE_CLIP, SAFE_FEATURE_CLIP, out=x)

    np.nan_to_num(y, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    if n_aux > 0:
        np.nan_to_num(y_aux, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

    if return_graph_event_ids:
        return x, y, y_aux, rows_possible, ptr, np.asarray(graph_event_ids, dtype=np.int32)
    return x, y, y_aux, rows_possible, ptr


def choose_rollout_sequence_indices(events, start_t: int, rollout_steps: int, max_samples: int, seed: int):
    counts = []
    total = 0
    for ev in events:
        n = min(ev["y"].shape[0], len(ev["rain"]))
        c = max(0, n - start_t - rollout_steps + 1)
        counts.append(c)
        total += c
    if max_samples <= 0 or total <= max_samples:
        picks = [None if c > 0 else np.array([], dtype=np.int32) for c in counts]
        return picks, total, total
    rng = np.random.RandomState(seed)
    picks = []
    kept = 0
    for c in counts:
        if c <= 0:
            picks.append(np.array([], dtype=np.int32))
            continue
        keep = max(1, int(round((c / max(total, 1)) * max_samples)))
        keep = min(keep, c)
        idx = rng.choice(c, size=keep, replace=False).astype(np.int32)
        picks.append(np.sort(idx))
        kept += keep
    return picks, total, kept


def build_rollout_samples(
    events,
    start_t: int,
    rollout_steps: int,
    y_lags: int,
    rain_lags: int,
    extra_cols,
    joint_aux_cols,
    extra_lags: int,
    max_samples: int,
    seed: int,
):
    if not events:
        return None, 0, 0
    n_extra = len(extra_cols)
    picks, samples_possible, _ = choose_rollout_sequence_indices(
        events, start_t=start_t, rollout_steps=rollout_steps, max_samples=max_samples, seed=seed
    )
    y_hist_list = []
    y_future_list = []
    rain_steps_list = []
    rain_mean_list = []
    extra_hist_list = []
    extra_future_list = []
    total = len(events)
    for j, ev in enumerate(events, 1):
        arr = ev["y"]
        rain = ev["rain"]
        extra = ev["extra"]
        n = min(arr.shape[0], len(rain))
        if n < start_t + rollout_steps:
            continue
        arr = arr[:n]
        rain = rain[:n]
        extra = {k: v[:n] for k, v in extra.items()}
        rain_hist = build_rain_lags(rain, rain_lags)
        rel = picks[j - 1]
        if rel is None:
            rel = np.arange(n - start_t - rollout_steps + 1, dtype=np.int32)
        if rel.size == 0:
            continue
        rain_mean = float(np.mean(rain))
        for r in rel.tolist():
            tt = start_t + int(r)
            y_hist = np.stack([arr[tt - 1 - k, :] for k in range(y_lags)], axis=0).astype(np.float32, copy=False)
            y_future = arr[tt : tt + rollout_steps, :].astype(np.float32, copy=False)
            rain_steps = rain_hist[tt : tt + rollout_steps, : rain_lags + 1].astype(np.float32, copy=False)
            y_hist_list.append(y_hist)
            y_future_list.append(y_future)
            rain_steps_list.append(rain_steps)
            rain_mean_list.append(rain_mean)
            if n_extra > 0 and extra_lags > 0:
                ex_hist_cols = []
                ex_future_cols = []
                for col in extra_cols:
                    ex = extra[col]
                    ex_hist_cols.append(np.stack([ex[tt - 1 - k, :] for k in range(extra_lags)], axis=0).astype(np.float32, copy=False))
                    ex_future_cols.append(ex[tt : tt + rollout_steps, :].astype(np.float32, copy=False))
                extra_hist_list.append(np.stack(ex_hist_cols, axis=0))
                extra_future_list.append(np.stack(ex_future_cols, axis=0))
        if j % max(1, LOG_EVERY_EVENTS) == 0 or j == total:
            log(f"rollout sample build progress: {j}/{total} events")
    if not y_hist_list:
        return None, samples_possible, 0
    data = {
        "y_hist": np.stack(y_hist_list, axis=0).astype(np.float32, copy=False),
        "y_future": np.stack(y_future_list, axis=0).astype(np.float32, copy=False),
        "rain_steps": np.stack(rain_steps_list, axis=0).astype(np.float32, copy=False),
        "rain_mean": np.asarray(rain_mean_list, dtype=np.float32),
        "count": int(len(y_hist_list)),
    }
    if n_extra > 0 and extra_lags > 0 and extra_hist_list:
        data["extra_hist"] = np.stack(extra_hist_list, axis=0).astype(np.float32, copy=False)
        data["extra_future"] = np.stack(extra_future_list, axis=0).astype(np.float32, copy=False)
    else:
        data["extra_hist"] = None
        data["extra_future"] = None
    return data, samples_possible, int(len(y_hist_list))


def fit_scaler(x: np.ndarray):
    mean = x.mean(axis=0, dtype=np.float64).astype(np.float32)
    std = x.std(axis=0, dtype=np.float64).astype(np.float32)
    std = np.where(std < 1e-6, 1.0, std).astype(np.float32)
    return mean, std


def apply_scaler(x: np.ndarray, mean: np.ndarray, std: np.ndarray):
    return ((x - mean[None, :]) / std[None, :]).astype(np.float32, copy=False)


def apply_scaler_torch(x: torch.Tensor, mean: torch.Tensor, std: torch.Tensor):
    return (x - mean.view(1, 1, -1)) / std.view(1, 1, -1)


# =========================
# Model
# =========================

def flat_to_graph_samples(x_flat: np.ndarray, y_flat: np.ndarray, n_nodes: int, tag: str):
    if len(y_flat) == 0:
        return (
            np.empty((0, n_nodes, x_flat.shape[1]), dtype=np.float32),
            np.empty((0, n_nodes), dtype=np.float32),
        )
    if len(y_flat) % n_nodes != 0:
        raise RuntimeError(f"{tag}: rows must be divisible by n_nodes: rows={len(y_flat)} n_nodes={n_nodes}")
    n_graph = len(y_flat) // n_nodes
    xg = x_flat.reshape(n_graph, n_nodes, x_flat.shape[1]).astype(np.float32, copy=False)
    yg = y_flat.reshape(n_graph, n_nodes).astype(np.float32, copy=False)
    return xg, yg


def flat_to_graph_aux_samples(y_aux_flat: np.ndarray, n_nodes: int, n_aux: int, tag: str):
    if n_aux <= 0:
        return None
    if y_aux_flat is None or y_aux_flat.size == 0:
        return np.empty((0, n_nodes, n_aux), dtype=np.float32)
    if y_aux_flat.ndim != 2 or y_aux_flat.shape[1] != n_aux:
        raise RuntimeError(f"{tag}: aux target shape mismatch: got {y_aux_flat.shape}, n_aux={n_aux}")
    if y_aux_flat.shape[0] % n_nodes != 0:
        raise RuntimeError(f"{tag}: aux rows must be divisible by n_nodes: rows={y_aux_flat.shape[0]} n_nodes={n_nodes}")
    n_graph = y_aux_flat.shape[0] // n_nodes
    return y_aux_flat.reshape(n_graph, n_nodes, n_aux).astype(np.float32, copy=False)


class PhysGraphLayer(nn.Module):
    def __init__(self, d_model: int, dropout: float):
        super().__init__()
        self.self_lin = nn.Linear(d_model, d_model)
        self.msg_lin = nn.Linear(d_model, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(d_model)
    def forward(self, h: torch.Tensor, edge_index: torch.Tensor):
        src, dst = edge_index
        msg = self.msg_lin(h[src])
        if h.dtype != msg.dtype:
            h = h.to(msg.dtype)
        agg = torch.zeros((h.shape[0], h.shape[1]), device=h.device, dtype=msg.dtype)
        agg.index_add_(0, dst, msg)
        deg = torch.zeros((h.shape[0], 1), device=h.device, dtype=msg.dtype)
        deg.index_add_(0, dst, torch.ones((dst.shape[0], 1), device=h.device, dtype=msg.dtype))
        agg = agg / deg.clamp_min(1.0)
        u = torch.relu(self.self_lin(h) + agg)
        h = self.norm1(h + u)
        h = self.norm2(h + self.ffn(h))
        return h


def edge_softmax_by_dst(scores: torch.Tensor, dst: torch.Tensor, num_nodes: int) -> torch.Tensor:
    if scores.ndim != 1:
        raise RuntimeError(f"edge softmax expects 1D scores, got shape={tuple(scores.shape)}")
    if dst.ndim != 1:
        raise RuntimeError(f"edge softmax expects 1D dst index, got shape={tuple(dst.shape)}")
    if scores.shape[0] != dst.shape[0]:
        raise RuntimeError(f"edge softmax size mismatch: scores={scores.shape[0]} dst={dst.shape[0]}")
    if scores.numel() == 0:
        return scores
    if hasattr(torch.Tensor, "scatter_reduce_"):
        max_per_dst = torch.full((num_nodes,), float("-inf"), device=scores.device, dtype=scores.dtype)
        max_per_dst.scatter_reduce_(0, dst, scores, reduce="amax", include_self=True)
        stable_scores = scores - max_per_dst[dst]
        exp_scores = torch.exp(stable_scores)
        denom = torch.zeros((num_nodes,), device=scores.device, dtype=exp_scores.dtype)
        denom.index_add_(0, dst, exp_scores)
        return exp_scores / denom[dst].clamp_min(1e-12)
    alpha = torch.zeros_like(scores)
    for node_idx in dst.unique(sorted=False).tolist():
        mask = dst == int(node_idx)
        alpha[mask] = torch.softmax(scores[mask], dim=0)
    return alpha


class PhysGraphAttentionLayer(nn.Module):
    def __init__(self, d_model: int, dropout: float, attn_dropout: float = 0.0, use_out_proj: bool = True):
        super().__init__()
        self.self_lin = nn.Linear(d_model, d_model)
        self.q_lin = nn.Linear(d_model, d_model)
        self.k_lin = nn.Linear(d_model, d_model)
        self.v_lin = nn.Linear(d_model, d_model)
        self.out_lin = nn.Linear(d_model, d_model) if use_out_proj else nn.Identity()
        self.attn_dropout = nn.Dropout(attn_dropout)
        self.scale = float(d_model) ** -0.5
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(d_model)
    def forward(self, h: torch.Tensor, edge_index: torch.Tensor):
        src, dst = edge_index
        q = self.q_lin(h)
        k = self.k_lin(h)
        v = self.v_lin(h)
        score = torch.sum(q[dst] * k[src], dim=-1) * self.scale
        alpha = edge_softmax_by_dst(score, dst, h.shape[0])
        alpha = self.attn_dropout(alpha)
        msg = v[src] * alpha.unsqueeze(-1)
        if h.dtype != msg.dtype:
            h = h.to(msg.dtype)
        agg = torch.zeros((h.shape[0], h.shape[1]), device=h.device, dtype=msg.dtype)
        agg.index_add_(0, dst, msg)
        agg = self.out_lin(agg)
        u = torch.relu(self.self_lin(h) + agg)
        h = self.norm1(h + u)
        h = self.norm2(h + self.ffn(h))
        return h


class RainFiLMGate(nn.Module):
    def __init__(self, rain_dim: int, d_model: int, hidden_dim: int, scale: float = 0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(rain_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, d_model * 2),
        )
        self.scale = float(scale)
    def forward(self, h_graph: torch.Tensor, rain_ctx: torch.Tensor) -> torch.Tensor:
        if rain_ctx.ndim != 2:
            raise RuntimeError(f"rain_ctx must be 2D, got shape={tuple(rain_ctx.shape)}")
        params = self.net(rain_ctx.to(dtype=h_graph.dtype))
        gamma, beta = params.chunk(2, dim=-1)
        gamma = torch.tanh(gamma) * self.scale
        beta = torch.tanh(beta) * self.scale
        return h_graph * (1.0 + gamma[:, None, :]) + beta[:, None, :]


class PhysicsInformedGNN(nn.Module):
    def __init__(
        self,
        n_features: int,
        n_nodes: int,
        edge_index_np: np.ndarray,
        d_model: int,
        n_layers: int,
        dropout: float,
        node_embed_dim: int,
        n_aux_targets: int = 0,
        layer_type: str = "attn",
        attn_dropout: float = 0.0,
        attn_use_out_proj: bool = True,
        y_lags: int = 10,
        rain_lags: int = 80,
        n_extra_cols: int = 0,
        extra_lags: int = 0,
        static_dim: int = 0,
        rain_gate_enable: bool = True,
        rain_gate_hidden: int = 64,
        rain_gate_scale: float = 0.5,
    ):
        super().__init__()
        self.n_nodes = int(n_nodes)
        self.node_embed_dim = max(0, int(node_embed_dim))
        self.node_emb = nn.Embedding(self.n_nodes, self.node_embed_dim) if self.node_embed_dim > 0 else None
        in_dim = n_features + self.node_embed_dim
        self.input_proj = nn.Linear(in_dim, d_model)
        layer_type = str(layer_type).strip().lower()
        if layer_type == "attn":
            self.layers = nn.ModuleList([
                PhysGraphAttentionLayer(d_model, dropout, attn_dropout=attn_dropout, use_out_proj=attn_use_out_proj)
                for _ in range(max(1, n_layers))
            ])
        elif layer_type == "mean":
            self.layers = nn.ModuleList([PhysGraphLayer(d_model, dropout) for _ in range(max(1, n_layers))])
        else:
            raise RuntimeError(f"unsupported layer_type: {layer_type}")
        self.layer_type = layer_type
        self.n_aux_targets = max(0, int(n_aux_targets))
        self.out_dim = 1 + self.n_aux_targets
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, self.out_dim))
        edge_index = torch.from_numpy(edge_index_np.astype(np.int64, copy=False))
        self.register_buffer("edge_index_base", edge_index, persistent=False)
        self.register_buffer("node_index_base", torch.arange(self.n_nodes, dtype=torch.long), persistent=False)
        self._edge_cache = {}
        self.beta_rain = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))
        extra_dim = int(n_extra_cols) * max(0, int(extra_lags))
        self.rain_start = 1 + int(static_dim) + int(y_lags) + int(extra_dim)
        self.rain_ctx_dim = int(rain_lags) + 2
        self.rain_gate_enable = bool(rain_gate_enable)
        if self.rain_gate_enable:
            self.rain_gate = RainFiLMGate(
                rain_dim=self.rain_ctx_dim,
                d_model=d_model,
                hidden_dim=max(8, int(rain_gate_hidden)),
                scale=rain_gate_scale,
            )
        else:
            self.rain_gate = None
    def _batched_edge_index(self, batch_size: int, device: torch.device):
        key = (int(batch_size), str(device))
        if key in self._edge_cache:
            return self._edge_cache[key]
        e = self.edge_index_base.to(device)
        offsets = (torch.arange(batch_size, device=device, dtype=torch.long) * self.n_nodes).view(batch_size, 1, 1)
        eb = e.view(1, 2, -1) + offsets
        eb = eb.permute(1, 0, 2).reshape(2, -1).contiguous()
        self._edge_cache[key] = eb
        return eb
    def forward(self, x_graph: torch.Tensor):
        b, n, _ = x_graph.shape
        if n != self.n_nodes:
            raise RuntimeError(f"node size mismatch: got {n}, expected {self.n_nodes}")
        if self.node_emb is not None:
            node_ids = self.node_index_base.to(x_graph.device).view(1, n).expand(b, n)
            emb = self.node_emb(node_ids)
            x = torch.cat([x_graph, emb], dim=-1)
        else:
            x = x_graph
        h = self.input_proj(x)
        if self.rain_gate is not None:
            rain_end = self.rain_start + self.rain_ctx_dim
            if rain_end > x_graph.shape[2]:
                raise RuntimeError(
                    f"invalid rain gate layout: rain_start={self.rain_start} rain_end={rain_end} total_dim={x_graph.shape[2]}"
                )
            rain_ctx = x_graph[:, 0, self.rain_start:rain_end]
            h = self.rain_gate(h, rain_ctx)
        h = h.reshape(b * n, -1)
        edge_index = self._batched_edge_index(b, x_graph.device)
        for layer in self.layers:
            h = layer(h, edge_index)
        out = self.head(h).reshape(b, n, self.out_dim)
        main_delta = out[:, :, 0]
        aux_delta = out[:, :, 1:] if self.n_aux_targets > 0 else None
        return main_delta, aux_delta



# =========================
# Train / Eval
# =========================

def make_loaders(x_train_g, y_train_g, x_val_g, y_val_g, batch_size: int,
                 sample_weight_train_g=None, y_train_aux_g=None, y_val_aux_g=None):

    if sample_weight_train_g is not None:
        wt = np.asarray(sample_weight_train_g, dtype=np.float32)
        if wt.shape[0] != x_train_g.shape[0]:
            raise RuntimeError(f"sample_weight_train_g size mismatch: {wt.shape[0]} vs n_graph={x_train_g.shape[0]}")

        if y_train_aux_g is not None:
            train_ds = TensorDataset(
                torch.from_numpy(x_train_g),
                torch.from_numpy(y_train_g),
                torch.from_numpy(y_train_aux_g),
                torch.from_numpy(wt),
            )
        else:
            train_ds = TensorDataset(torch.from_numpy(x_train_g), torch.from_numpy(y_train_g), torch.from_numpy(wt))
    else:
        if y_train_aux_g is not None:
            train_ds = TensorDataset(torch.from_numpy(x_train_g), torch.from_numpy(y_train_g), torch.from_numpy(y_train_aux_g))
        else:
            train_ds = TensorDataset(torch.from_numpy(x_train_g), torch.from_numpy(y_train_g))

    if y_val_aux_g is not None:
        val_ds = TensorDataset(torch.from_numpy(x_val_g), torch.from_numpy(y_val_g), torch.from_numpy(y_val_aux_g))
    else:
        val_ds = TensorDataset(torch.from_numpy(x_val_g), torch.from_numpy(y_val_g))

    pin = DEVICE == "cuda"
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin, drop_last=False)
    return train_loader, val_loader


def compute_physics_losses(pred: torch.Tensor, x_graph: torch.Tensor, model: PhysicsInformedGNN,
                          y_lags: int, rain_lags: int, extra_cols, extra_lags: int):

    b, n = pred.shape
    pred_flat = pred.reshape(b * n)

    edge_index = model._batched_edge_index(b, pred.device)
    src, dst = edge_index
    smooth = torch.mean((pred_flat[src] - pred_flat[dst]) ** 2)

    extra_dim = len(extra_cols) * max(0, int(extra_lags))
    fixed_dim = 1 + int(y_lags) + int(extra_dim) + (int(rain_lags) + 1) + 1
    static_dim = int(x_graph.shape[2]) - fixed_dim
    if static_dim < 0:
        raise RuntimeError(
            f"invalid feature layout: total_dim={int(x_graph.shape[2])} fixed_dim={fixed_dim} static_dim={static_dim}"
        )
    y_start = 1 + static_dim

    if y_lags >= 2:
        y_prev = x_graph[:, :, y_start]
        y_prev2 = x_graph[:, :, y_start + 1]
        prev_delta = y_prev - y_prev2
        dyn = nn.functional.mse_loss(pred, prev_delta)
    else:
        dyn = torch.zeros((), device=pred.device, dtype=pred.dtype)

    rain_start = y_start + int(y_lags) + int(extra_dim)
    rain_now = x_graph[:, 0, rain_start]
    mass = torch.mean((pred.mean(dim=1) - model.beta_rain * rain_now) ** 2)

    return smooth, mass, dyn


def compute_rollout_loss_batch(
    model: PhysicsInformedGNN,
    y_hist: torch.Tensor,
    y_future: torch.Tensor,
    extra_hist: torch.Tensor | None,
    extra_future: torch.Tensor | None,
    rain_steps: torch.Tensor,
    rain_mean: torch.Tensor,
    scaler_mean_t: torch.Tensor,
    scaler_std_t: torch.Tensor,
    static_feat_t: torch.Tensor | None,
    y_lags: int,
    rain_lags: int,
    extra_cols,
    joint_aux_cols,
    extra_lags: int,
):
    b, _, n_nodes = y_hist.shape
    device = y_hist.device
    dtype = y_hist.dtype
    node_pos = torch.arange(n_nodes, device=device, dtype=dtype) / max(1.0, float(n_nodes - 1))
    node_pos = node_pos.view(1, n_nodes, 1).expand(b, n_nodes, 1)
    static_part = None
    if static_feat_t is not None and static_feat_t.shape[1] > 0:
        static_part = static_feat_t.to(device=device, dtype=dtype).unsqueeze(0).expand(b, -1, -1)
    joint_aux_map = {col: idx for idx, col in enumerate(joint_aux_cols)}
    y_hist_cur = y_hist
    extra_hist_cur = extra_hist
    losses = []
    horizon = int(y_future.shape[1])
    for step in range(horizon):
        feat_parts = [node_pos]
        if static_part is not None:
            feat_parts.append(static_part)
        feat_parts.append(y_hist_cur.permute(0, 2, 1))
        if extra_hist_cur is not None and extra_lags > 0:
            extra_parts = [extra_hist_cur[:, col_idx].permute(0, 2, 1) for col_idx in range(extra_hist_cur.shape[1])]
            if extra_parts:
                feat_parts.append(torch.cat(extra_parts, dim=-1))
        step_r = rain_steps[:, step].unsqueeze(1).expand(b, n_nodes, rain_lags + 1)
        step_mean = rain_mean.view(b, 1, 1).expand(b, n_nodes, 1)
        feat_parts.extend([step_r, step_mean])
        x_step = torch.cat(feat_parts, dim=-1)
        x_step = apply_scaler_torch(x_step, scaler_mean_t, scaler_std_t)
        pred_delta, pred_aux = model(x_step)
        y_next = y_hist_cur[:, 0, :] + pred_delta
        y_next = torch.nan_to_num(y_next, nan=0.0, posinf=SAFE_LEVEL_CLIP, neginf=-SAFE_LEVEL_CLIP)
        y_next = y_next.clamp(-SAFE_LEVEL_CLIP, SAFE_LEVEL_CLIP)
        losses.append(nn.functional.mse_loss(y_next, y_future[:, step, :]))
        y_hist_cur = torch.cat([y_next.unsqueeze(1), y_hist_cur[:, :-1, :]], dim=1)
        if extra_hist_cur is not None and extra_lags > 0:
            next_cols = []
            for col_idx, col in enumerate(extra_cols):
                prev_ex = extra_hist_cur[:, col_idx, 0, :]
                if col in joint_aux_map and pred_aux is not None:
                    ex_next = prev_ex + pred_aux[:, :, joint_aux_map[col]]
                elif extra_future is not None:
                    ex_next = extra_future[:, col_idx, step, :]
                else:
                    ex_next = prev_ex
                ex_next = torch.nan_to_num(ex_next, nan=0.0, posinf=SAFE_FEATURE_CLIP, neginf=-SAFE_FEATURE_CLIP)
                ex_next = ex_next.clamp(-SAFE_FEATURE_CLIP, SAFE_FEATURE_CLIP)
                next_cols.append(torch.cat([ex_next.unsqueeze(1), extra_hist_cur[:, col_idx, :-1, :]], dim=1))
            extra_hist_cur = torch.stack(next_cols, dim=1)
    return torch.stack(losses).mean() if losses else torch.zeros((), device=device, dtype=dtype)


def evaluate_rmse(model, loader):
    model.eval()
    sse = 0.0
    cnt = 0
    with torch.no_grad():
        for batch in loader:
            xb = batch[0].to(DEVICE, non_blocking=True)
            yb = batch[1].to(DEVICE, non_blocking=True)
            pred, _ = model(xb)
            diff = pred - yb
            sse += float(torch.sum(diff * diff).item())
            cnt += int(yb.numel())
    return float(np.sqrt(sse / max(cnt, 1))) if cnt > 0 else float("inf")


def create_model(
    n_features: int,
    n_nodes: int,
    edge_index_np: np.ndarray,
    n_aux_targets: int,
    n_extra_cols: int,
    static_dim: int,
):
    return PhysicsInformedGNN(
        n_features=n_features,
        n_nodes=n_nodes,
        edge_index_np=edge_index_np,
        d_model=GNN_D_MODEL,
        n_layers=GNN_N_LAYERS,
        dropout=GNN_DROPOUT,
        node_embed_dim=NODE_EMBED_DIM,
        n_aux_targets=n_aux_targets,
        layer_type=GNN_LAYER_TYPE,
        attn_dropout=ATTN_DROPOUT,
        attn_use_out_proj=ATTN_USE_OUT_PROJ,
        y_lags=Y_LAGS,
        rain_lags=RAIN_LAGS,
        n_extra_cols=n_extra_cols,
        extra_lags=EXTRA_LAGS,
        static_dim=static_dim,
        rain_gate_enable=RAIN_GATE_ENABLE,
        rain_gate_hidden=RAIN_GATE_HIDDEN,
        rain_gate_scale=RAIN_GATE_SCALE,
    ).to(DEVICE)


def train_one_fold(
    x_train, y_train, y_train_aux,
    x_val, y_val, y_val_aux,
    fold_idx: int,
    n_nodes: int,
    edge_index_np: np.ndarray,
    extra_cols,
    joint_aux_cols,
    extra_lags: int,
    sample_weight_train_graph=None,
    epoch_oof_eval_every: int = 0,
    epoch_oof_eval_fn=None,
    aux_loss_weight: float = 1.0,
    rollout_train_data=None,
    scaler_mean: np.ndarray | None = None,
    scaler_std: np.ndarray | None = None,
    static_feat: np.ndarray | None = None,
):
    x_train_g, y_train_g = flat_to_graph_samples(x_train, y_train, n_nodes, tag=f"fold{fold_idx+1}:train")
    x_val_g, y_val_g = flat_to_graph_samples(x_val, y_val, n_nodes, tag=f"fold{fold_idx+1}:val")

    n_aux = len(joint_aux_cols)
    y_train_aux_g = flat_to_graph_aux_samples(y_train_aux, n_nodes, n_aux, tag=f"fold{fold_idx+1}:train")
    y_val_aux_g = flat_to_graph_aux_samples(y_val_aux, n_nodes, n_aux, tag=f"fold{fold_idx+1}:val")

    train_loader, val_loader = make_loaders(
        x_train_g, y_train_g, x_val_g, y_val_g,
        GRAPH_BATCH_SIZE,
        sample_weight_train_g=sample_weight_train_graph,
        y_train_aux_g=y_train_aux_g,
        y_val_aux_g=y_val_aux_g,
    )

    model = create_model(
        n_features=x_train_g.shape[2],
        n_nodes=n_nodes,
        edge_index_np=edge_index_np,
        n_aux_targets=n_aux,
        n_extra_cols=len(extra_cols),
        static_dim=0 if static_feat is None else int(static_feat.shape[1]),
    )

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=MIN_LR)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_rmse = float("inf")
    best_state = None
    best_epoch_oof_std = float("inf")
    best_oof_state = None
    patience_left = PATIENCE

    rollout_enabled = (
        ROLLOUT_LOSS_ENABLE
        and rollout_train_data is not None
        and int(rollout_train_data.get("count", 0)) > 0
        and scaler_mean is not None
        and scaler_std is not None
    )
    if rollout_enabled:
        scaler_mean_t = torch.from_numpy(scaler_mean.astype(np.float32, copy=False)).to(DEVICE)
        scaler_std_t = torch.from_numpy(scaler_std.astype(np.float32, copy=False)).to(DEVICE)
        static_feat_t = None if static_feat is None else torch.from_numpy(static_feat.astype(np.float32, copy=False)).to(DEVICE)
    else:
        scaler_mean_t = None
        scaler_std_t = None
        static_feat_t = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_sse = 0.0
        train_cnt = 0
        rollout_loss_mean = float("nan")

        for batch in train_loader:
            xb, yb = batch[0], batch[1]
            third = batch[2] if len(batch) >= 3 else None
            fourth = batch[3] if len(batch) >= 4 else None

            yb_aux = None
            wb = None
            if len(batch) == 4:
                yb_aux = third
                wb = fourth
            elif len(batch) == 3:
                if getattr(third, "ndim", 0) >= 3:
                    yb_aux = third
                else:
                    wb = third

            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            if yb_aux is not None:
                yb_aux = yb_aux.to(DEVICE, non_blocking=True)
            if wb is not None:
                wb = wb.to(DEVICE, non_blocking=True).float()

            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                pred, pred_aux = model(xb)

                if wb is None:
                    loss_main = nn.functional.mse_loss(pred, yb)
                else:
                    per_graph = torch.mean((pred - yb) ** 2, dim=1)
                    loss_main = torch.sum(per_graph * wb) / torch.clamp(wb.sum(), min=1e-8)

                if yb_aux is not None and pred_aux is not None:
                    loss_aux = nn.functional.mse_loss(pred_aux, yb_aux)
                else:
                    loss_aux = torch.zeros((), device=pred.device, dtype=pred.dtype)

                loss_smooth, loss_mass, loss_dyn = compute_physics_losses(
                    pred=pred,
                    x_graph=xb,
                    model=model,
                    y_lags=Y_LAGS,
                    rain_lags=RAIN_LAGS,
                    extra_cols=extra_cols,
                    extra_lags=extra_lags,
                )

                loss = (
                    loss_main
                    + float(aux_loss_weight) * loss_aux
                    + float(LAMBDA_SMOOTH) * loss_smooth
                    + float(LAMBDA_MASS) * loss_mass
                    + float(LAMBDA_DYN) * loss_dyn
                )

            scaler.scale(loss).backward()
            if GRAD_CLIP_NORM > 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(opt)
            scaler.update()

            diff = pred.detach() - yb
            train_sse += float(torch.sum(diff * diff).item())
            train_cnt += int(yb.numel())

        if rollout_enabled:
            perm = np.random.permutation(int(rollout_train_data["count"]))
            rollout_loss_sum = 0.0
            rollout_batches = 0
            for st in range(0, len(perm), ROLLOUT_BATCH_SIZE):
                idx = perm[st : st + ROLLOUT_BATCH_SIZE]
                y_hist = torch.from_numpy(rollout_train_data["y_hist"][idx]).to(DEVICE, non_blocking=True)
                y_future = torch.from_numpy(rollout_train_data["y_future"][idx]).to(DEVICE, non_blocking=True)
                rain_steps = torch.from_numpy(rollout_train_data["rain_steps"][idx]).to(DEVICE, non_blocking=True)
                rain_mean = torch.from_numpy(rollout_train_data["rain_mean"][idx]).to(DEVICE, non_blocking=True)
                extra_hist = None
                extra_future = None
                if rollout_train_data.get("extra_hist") is not None:
                    extra_hist = torch.from_numpy(rollout_train_data["extra_hist"][idx]).to(DEVICE, non_blocking=True)
                    extra_future = torch.from_numpy(rollout_train_data["extra_future"][idx]).to(DEVICE, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=USE_AMP):
                    rollout_loss = compute_rollout_loss_batch(
                        model=model,
                        y_hist=y_hist,
                        y_future=y_future,
                        extra_hist=extra_hist,
                        extra_future=extra_future,
                        rain_steps=rain_steps,
                        rain_mean=rain_mean,
                        scaler_mean_t=scaler_mean_t,
                        scaler_std_t=scaler_std_t,
                        static_feat_t=static_feat_t,
                        y_lags=Y_LAGS,
                        rain_lags=RAIN_LAGS,
                        extra_cols=extra_cols,
                        joint_aux_cols=joint_aux_cols,
                        extra_lags=extra_lags,
                    )
                    loss_rollout = float(ROLLOUT_WEIGHT) * rollout_loss
                scaler.scale(loss_rollout).backward()
                if GRAD_CLIP_NORM > 0:
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(opt)
                scaler.update()
                rollout_loss_sum += float(rollout_loss.item())
                rollout_batches += 1
            rollout_loss_mean = rollout_loss_sum / max(rollout_batches, 1)

        train_rmse = float(np.sqrt(train_sse / max(train_cnt, 1)))
        val_rmse = evaluate_rmse(model, val_loader)
        scheduler.step(val_rmse)
        cur_lr = float(opt.param_groups[0]["lr"])
        std = float(STD_DEV[TARGET])
        rollout_log = f" rollout_mse={rollout_loss_mean:.6f}" if np.isfinite(rollout_loss_mean) else ""

        log(
            f"fold={fold_idx+1} epoch={epoch}/{EPOCHS} "
            f"train_rmse={train_rmse:.6f} val_rmse={val_rmse:.6f} val_std={val_rmse/std:.6f} "
            f"lr={cur_lr:.2e}{rollout_log}"
        )

        if epoch_oof_eval_fn is not None and epoch_oof_eval_every > 0 and (epoch % epoch_oof_eval_every == 0):
            try:
                epoch_oof_std = float(epoch_oof_eval_fn(model))
                log(f"fold={fold_idx+1} epoch={epoch}/{EPOCHS} oof_lb_std@epoch={epoch_oof_std:.6f}")
                if np.isfinite(epoch_oof_std) and epoch_oof_std < best_epoch_oof_std:
                    best_epoch_oof_std = epoch_oof_std
                    best_oof_state = copy.deepcopy(model.state_dict())
            except Exception as exc:
                log(f"fold={fold_idx+1} epoch={epoch}/{EPOCHS} oof_lb_eval_skipped: {exc}")

        if val_rmse < best_rmse:
            best_rmse = val_rmse
            best_state = copy.deepcopy(model.state_dict())
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                log(f"fold={fold_idx+1} early_stop")
                break

    if best_state is None:
        raise RuntimeError("best_state is None (training failed?)")

    model.load_state_dict(best_state)
    return {
        "model": model,
        "best_val_rmse": float(best_rmse),
        "best_val_state": best_state,
        "best_epoch_oof_std": float(best_epoch_oof_std) if np.isfinite(best_epoch_oof_std) else float("nan"),
        "best_oof_state": best_oof_state,
    }


def init_parallel_context(**kwargs):
    global _PARALLEL_CONTEXT
    _PARALLEL_CONTEXT = dict(kwargs)


def run_fold_training_job(job: dict):
    fold_idx = int(job["fold_idx"])
    run_i = int(job["run_i"])
    total_runs = int(job["total_runs"])
    gpu_id = job.get("gpu_id")
    worker_started_at = time.time()
    worker_pid = os.getpid()
    set_runtime_device(gpu_id)
    set_seed(SEED + fold_idx * 1000 + TARGET_MODEL_ID * 10 + TARGET_NODE_TYPE)

    val_ids = list(job["val_ids"])
    tr_ids = list(job["tr_ids"])
    train_events_all = _PARALLEL_CONTEXT["train_events_all"]
    node_ids = _PARALLEL_CONTEXT["node_ids"]
    edge_index_np = _PARALLEL_CONTEXT["edge_index_np"]
    extra_cols_active = _PARALLEL_CONTEXT["extra_cols_active"]
    joint_aux_cols_active = _PARALLEL_CONTEXT["joint_aux_cols_active"]
    static_feat = _PARALLEL_CONTEXT["static_feat"]
    static_cols_active = _PARALLEL_CONTEXT["static_cols_active"]

    log(f"fold {fold_idx+1}/{N_FOLDS} (run {run_i}/{total_runs}) worker_start pid={worker_pid} gpu={gpu_id} train_events={len(tr_ids)} val_events={len(val_ids)}")

    tr_events = [
        {"event_id": ev["event_id"], "t": ev["t"], "y": ev["y"].copy(), "rain": ev["rain"], "extra": {k: v.copy() for k, v in ev["extra"].items()}}
        for ev in (train_events_all[eid] for eid in tr_ids)
    ]
    va_events = [
        {"event_id": ev["event_id"], "t": ev["t"], "y": ev["y"].copy(), "rain": ev["rain"], "extra": {k: v.copy() for k, v in ev["extra"].items()}}
        for ev in (train_events_all[eid] for eid in val_ids)
    ]

    means = compute_node_means_from_events(tr_events)
    extra_means = compute_extra_means_from_events(tr_events, extra_cols_active)
    fill_missing(tr_events, means, extra_means)
    fill_missing(va_events, means, extra_means)

    seed_base = SEED + fold_idx * 100

    if RAIN3_ENABLED:
        x_train, y_train, y_train_aux, rows_pos_train, rows_kept_train, graph_event_ids_train = build_samples(
            tr_events, START_T, Y_LAGS, RAIN_LAGS, extra_cols_active, joint_aux_cols_active, EXTRA_LAGS, static_feat, MAX_TRAIN_ROWS, seed_base,
            return_graph_event_ids=True,
        )
    else:
        x_train, y_train, y_train_aux, rows_pos_train, rows_kept_train = build_samples(
            tr_events, START_T, Y_LAGS, RAIN_LAGS, extra_cols_active, joint_aux_cols_active, EXTRA_LAGS, static_feat, MAX_TRAIN_ROWS, seed_base,
            return_graph_event_ids=False,
        )
        graph_event_ids_train = None

    x_val, y_val, y_val_aux, rows_pos_val, rows_kept_val = build_samples(
        va_events, START_T, Y_LAGS, RAIN_LAGS, extra_cols_active, joint_aux_cols_active, EXTRA_LAGS, static_feat, MAX_VAL_ROWS, seed_base + 1,
        return_graph_event_ids=False,
    )

    rollout_train_data, rollout_pos_train, rollout_kept_train = build_rollout_samples(
        tr_events,
        start_t=START_T,
        rollout_steps=ROLLOUT_STEPS,
        y_lags=Y_LAGS,
        rain_lags=RAIN_LAGS,
        extra_cols=extra_cols_active,
        joint_aux_cols=joint_aux_cols_active,
        extra_lags=EXTRA_LAGS,
        max_samples=ROLLOUT_MAX_SAMPLES,
        seed=seed_base + 11,
    )

    log(f"fold {fold_idx+1}: train_samples={rows_kept_train}/{rows_pos_train} val_samples={rows_kept_val}/{rows_pos_val} feat_dim={x_train.shape[1]} rollout_samples={rollout_kept_train}/{rollout_pos_train} gpu={gpu_id}")

    scaler_mean, scaler_std = fit_scaler(x_train)
    x_train = apply_scaler(x_train, scaler_mean, scaler_std)
    x_val = apply_scaler(x_val, scaler_mean, scaler_std)

    anchor_models = None
    anchor_params = None
    selected_checkpoint = "val"
    model = None
    best_rmse = float("nan")
    n_aux_targets = len(joint_aux_cols_active)
    n_features = int(x_train.shape[1])

    if RAIN3_ENABLED:
        anchor_params = build_rain_anchor_params(tr_events)
        graph_sample_w = build_anchor_graph_weights(
            graph_event_ids=graph_event_ids_train,
            score_by_event=anchor_params["score_by_event"],
            centers=anchor_params["centers"],
            bandwidth=anchor_params["bandwidth"],
        )
        log(
            "rain3 anchors: "
            f"centers=[{anchor_params['centers'][0]:.4f}, {anchor_params['centers'][1]:.4f}, {anchor_params['centers'][2]:.4f}] "
            f"bandwidth={anchor_params['bandwidth']:.4f}"
        )

        anchor_models = []
        anchor_best_rmses = []

        for anchor_idx, anchor_name in enumerate(("dry", "mid", "wet")):
            w_anchor = graph_sample_w[:, anchor_idx].astype(np.float32, copy=False)
            use_w = w_anchor if float(np.sum(w_anchor)) > 0.0 else None
            if use_w is None:
                log(f"anchor fallback: fold={fold_idx+1} name={anchor_name} (no positive weights)")

            log(f"anchor train: fold={fold_idx+1}/{N_FOLDS} name={anchor_name} graph_samples={x_train.shape[0] // max(1, len(node_ids))} gpu={gpu_id}")

            fit_result_k = train_one_fold(
                x_train, y_train, y_train_aux,
                x_val, y_val, y_val_aux,
                fold_idx,
                n_nodes=len(node_ids),
                edge_index_np=edge_index_np,
                extra_cols=extra_cols_active,
                joint_aux_cols=joint_aux_cols_active,
                extra_lags=EXTRA_LAGS,
                sample_weight_train_graph=use_w,
                epoch_oof_eval_every=0,
                epoch_oof_eval_fn=None,
                aux_loss_weight=AUX_LOSS_WEIGHT,
                rollout_train_data=rollout_train_data,
                scaler_mean=scaler_mean,
                scaler_std=scaler_std,
                static_feat=static_feat,
            )
            model_k = fit_result_k["model"]
            best_rmse_k = float(fit_result_k["best_val_rmse"])
            anchor_models.append(model_k)
            anchor_best_rmses.append(best_rmse_k)
            log(f"anchor done: fold={fold_idx+1}/{N_FOLDS} name={anchor_name} sample_std={best_rmse_k/float(STD_DEV[TARGET]):.6f} gpu={gpu_id}")

        model = anchor_models[1]
        best_rmse = float(np.mean(anchor_best_rmses))
        oof_lb_score, _ = evaluate_oof_lb_equivalent_weighted(
            events=va_events,
            models=anchor_models,
            scaler_mean=scaler_mean,
            scaler_std=scaler_std,
            anchor_params=anchor_params,
            target=TARGET,
            start_t=START_T,
            y_lags=Y_LAGS,
            rain_lags=RAIN_LAGS,
            extra_cols=extra_cols_active,
            joint_aux_cols=joint_aux_cols_active,
            extra_lags=EXTRA_LAGS,
            static_feat=static_feat,
        ) if OOF_LB_EVAL else (float("nan"), None)
        anchor_model_states = [state_dict_to_cpu(m.state_dict()) for m in anchor_models]
        model_state = state_dict_to_cpu(model.state_dict())
        del graph_sample_w
    else:
        epoch_eval_events = va_events[:EPOCH_OOF_MAX_EVENTS] if EPOCH_OOF_MAX_EVENTS > 0 else va_events
        if OOF_LB_EVAL and EPOCH_OOF_EVAL_EVERY > 0 and len(epoch_eval_events) > 0:
            def _epoch_oof_eval_fn(model_cur):
                score, _ = evaluate_oof_lb_equivalent(
                    events=epoch_eval_events,
                    model=model_cur,
                    scaler_mean=scaler_mean,
                    scaler_std=scaler_std,
                    target=TARGET,
                    start_t=START_T,
                    y_lags=Y_LAGS,
                    rain_lags=RAIN_LAGS,
                    extra_cols=extra_cols_active,
                    joint_aux_cols=joint_aux_cols_active,
                    extra_lags=EXTRA_LAGS,
                    static_feat=static_feat,
                )
                return float(score)
        else:
            _epoch_oof_eval_fn = None

        fit_result = train_one_fold(
            x_train, y_train, y_train_aux,
            x_val, y_val, y_val_aux,
            fold_idx,
            n_nodes=len(node_ids),
            edge_index_np=edge_index_np,
            extra_cols=extra_cols_active,
            joint_aux_cols=joint_aux_cols_active,
            extra_lags=EXTRA_LAGS,
            sample_weight_train_graph=None,
            epoch_oof_eval_every=EPOCH_OOF_EVAL_EVERY,
            epoch_oof_eval_fn=_epoch_oof_eval_fn,
            aux_loss_weight=AUX_LOSS_WEIGHT,
            rollout_train_data=rollout_train_data,
            scaler_mean=scaler_mean,
            scaler_std=scaler_std,
            static_feat=static_feat,
        )
        model = fit_result["model"]
        best_rmse = float(fit_result["best_val_rmse"])
        best_val_state = fit_result["best_val_state"]
        best_epoch_oof_std = float(fit_result["best_epoch_oof_std"])
        best_oof_state = fit_result["best_oof_state"]

        if OOF_LB_EVAL:
            val_checkpoint_oof = float("nan")
            epoch_checkpoint_oof = float("nan")

            model.load_state_dict(best_val_state)
            val_checkpoint_oof, _ = evaluate_oof_lb_equivalent(
                events=va_events,
                model=model,
                scaler_mean=scaler_mean,
                scaler_std=scaler_std,
                target=TARGET,
                start_t=START_T,
                y_lags=Y_LAGS,
                rain_lags=RAIN_LAGS,
                extra_cols=extra_cols_active,
                joint_aux_cols=joint_aux_cols_active,
                extra_lags=EXTRA_LAGS,
                static_feat=static_feat,
            )

            selected_checkpoint = "val"
            selected_state = best_val_state
            oof_lb_score = float(val_checkpoint_oof)

            if best_oof_state is not None:
                model.load_state_dict(best_oof_state)
                epoch_checkpoint_oof, _ = evaluate_oof_lb_equivalent(
                    events=va_events,
                    model=model,
                    scaler_mean=scaler_mean,
                    scaler_std=scaler_std,
                    target=TARGET,
                    start_t=START_T,
                    y_lags=Y_LAGS,
                    rain_lags=RAIN_LAGS,
                    extra_cols=extra_cols_active,
                    joint_aux_cols=joint_aux_cols_active,
                    extra_lags=EXTRA_LAGS,
                    static_feat=static_feat,
                )
                if np.isfinite(epoch_checkpoint_oof) and (not np.isfinite(oof_lb_score) or epoch_checkpoint_oof < oof_lb_score):
                    selected_checkpoint = "epoch_oof"
                    selected_state = best_oof_state
                    oof_lb_score = float(epoch_checkpoint_oof)

            model.load_state_dict(selected_state)
            tracked_epoch_oof_std = best_epoch_oof_std if np.isfinite(best_epoch_oof_std) else float("nan")
            log(
                f"fold {fold_idx+1} checkpoint_select: val_best_std={val_checkpoint_oof:.6f} "
                f"epoch_oof_best_std={epoch_checkpoint_oof:.6f} "
                f"epoch_oof_tracked={tracked_epoch_oof_std:.6f} chosen={selected_checkpoint} gpu={gpu_id}"
            )
        else:
            oof_lb_score = float("nan")

        anchor_model_states = None
        model_state = state_dict_to_cpu(model.state_dict())

    result = {
        "fold_idx": fold_idx,
        "val_ids": list(val_ids),
        "sample_rmse": float(best_rmse),
        "oof_lb_score": float(oof_lb_score),
        "model_state": model_state,
        "anchor_model_states": anchor_model_states,
        "anchor_params": anchor_params,
        "scaler_mean": scaler_mean,
        "scaler_std": scaler_std,
        "node_means": means,
        "extra_means": extra_means,
        "selected_checkpoint": selected_checkpoint if not RAIN3_ENABLED else "val",
        "static_cols": list(static_cols_active),
        "static_dim": int(static_feat.shape[1]),
        "n_extra_cols": int(len(extra_cols_active)),
        "n_features": n_features,
        "n_aux_targets": n_aux_targets,
        "n_nodes": int(len(node_ids)),
        "gpu_id": gpu_id,
    }

    del x_train, y_train, y_train_aux, x_val, y_val, y_val_aux, rollout_train_data, tr_events, va_events
    if graph_event_ids_train is not None:
        del graph_event_ids_train
    gc.collect()
    if _CUDA_RUNTIME_ENABLED:
        torch.cuda.empty_cache()
    return result


def restore_fold_state_from_result(result: dict, edge_index_np: np.ndarray):
    n_nodes = int(result["n_nodes"])
    n_features = int(result["n_features"])
    n_aux_targets = int(result["n_aux_targets"])
    n_extra_cols = int(result.get("n_extra_cols", 0))
    static_dim = int(result.get("static_dim", 0))

    if result.get("anchor_model_states") is not None:
        anchor_models = []
        for state_dict in result["anchor_model_states"]:
            m = create_model(
                n_features=n_features,
                n_nodes=n_nodes,
                edge_index_np=edge_index_np,
                n_aux_targets=n_aux_targets,
                n_extra_cols=n_extra_cols,
                static_dim=static_dim,
            )
            m.load_state_dict(state_dict)
            anchor_models.append(m)
        model = anchor_models[1]
    else:
        anchor_models = None
        model = create_model(
            n_features=n_features,
            n_nodes=n_nodes,
            edge_index_np=edge_index_np,
            n_aux_targets=n_aux_targets,
            n_extra_cols=n_extra_cols,
            static_dim=static_dim,
        )
        model.load_state_dict(result["model_state"])

    return {
        "fold_idx": result["fold_idx"],
        "val_ids": list(result["val_ids"]),
        "model": model,
        "scaler_mean": result["scaler_mean"],
        "scaler_std": result["scaler_std"],
        "selected_checkpoint": result.get("selected_checkpoint", "val"),
        "node_means": result["node_means"],
        "extra_means": result["extra_means"],
        "anchor_models": anchor_models,
        "anchor_params": result.get("anchor_params"),
        "static_cols": list(result.get("static_cols", [])),
        "static_dim": int(result.get("static_dim", 0)),
    }


def predict_delta(model, x_np: np.ndarray, batch_size: int):
    # x_np: [N,F] or [B,N,F]
    if x_np.ndim == 2:
        x_np = x_np[None, :, :]
        squeeze = True
    else:
        squeeze = False

    model.eval()
    out = np.empty((x_np.shape[0], x_np.shape[1]), dtype=np.float32)
    out_aux = None

    with torch.no_grad():
        for i in range(0, x_np.shape[0], batch_size):
            xb = torch.from_numpy(x_np[i : i + batch_size]).to(DEVICE)
            yp, yp_aux = model(xb)
            yp = yp.detach().cpu().numpy().astype(np.float32, copy=False)
            out[i : i + batch_size, :] = yp

            if yp_aux is not None:
                yp_aux = yp_aux.detach().cpu().numpy().astype(np.float32, copy=False)
                if out_aux is None:
                    out_aux = np.empty((x_np.shape[0], x_np.shape[1], yp_aux.shape[2]), dtype=np.float32)
                out_aux[i : i + batch_size, :, :] = yp_aux

    if squeeze:
        return out[0], (out_aux[0] if out_aux is not None else None)
    return out, out_aux


def predict_event_recursive(ev, model, scaler_mean, scaler_std,
                            start_t: int, y_lags: int, rain_lags: int,
                            extra_cols, joint_aux_cols, extra_lags: int, static_feat: np.ndarray | None):
    t = ev["t"]
    y = ev["y"].copy()
    rain = ev["rain"]

    n = min(len(t), y.shape[0], len(rain))
    t = t[:n]
    y = y[:n]
    rain = rain[:n]

    extra = {k: v[:n].copy() for k, v in ev["extra"].items()}

    rain_hist = build_rain_lags(rain, rain_lags)
    n_nodes = y.shape[1]
    node_pos = np.arange(n_nodes, dtype=np.float32) / max(1.0, float(n_nodes - 1))
    rain_mean = float(np.mean(rain)) if n > 0 else 0.0

    for tt in range(start_t, n):
        step_y = np.stack([y[tt - 1 - k, :] for k in range(y_lags)], axis=1).astype(np.float32, copy=False)

        step_extra = []
        if extra_lags > 0:
            for col in extra_cols:
                ex = extra[col]
                ex_lag = np.stack([ex[tt - 1 - k, :] for k in range(extra_lags)], axis=1).astype(np.float32, copy=False)
                step_extra.append(ex_lag)

        step_r = np.tile(rain_hist[tt, : rain_lags + 1][None, :], (n_nodes, 1)).astype(np.float32, copy=False)
        step_mean = np.full((n_nodes, 1), rain_mean, dtype=np.float32)

        feat_parts = [node_pos[:, None]]
        if static_feat is not None and static_feat.shape[1] > 0:
            feat_parts.append(static_feat)
        feat_parts.append(step_y)
        if step_extra:
            feat_parts.append(np.concatenate(step_extra, axis=1))
        feat_parts.extend([step_r, step_mean])

        feat = np.concatenate(feat_parts, axis=1)
        np.nan_to_num(feat, copy=False, nan=0.0, posinf=SAFE_FEATURE_CLIP, neginf=-SAFE_FEATURE_CLIP)
        np.clip(feat, -SAFE_FEATURE_CLIP, SAFE_FEATURE_CLIP, out=feat)

        feat = apply_scaler(feat, scaler_mean, scaler_std)

        bsz = max(1, PRED_BATCH_SIZE // max(1, n_nodes))
        delta, delta_aux = predict_delta(model, feat, batch_size=bsz)

        np.nan_to_num(delta, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        np.clip(delta, -SAFE_DELTA_CLIP, SAFE_DELTA_CLIP, out=delta)

        y[tt, :] = y[tt - 1, :] + delta
        np.nan_to_num(y[tt, :], copy=False, nan=0.0, posinf=SAFE_LEVEL_CLIP, neginf=-SAFE_LEVEL_CLIP)
        np.clip(y[tt, :], -SAFE_LEVEL_CLIP, SAFE_LEVEL_CLIP, out=y[tt, :])

        if delta_aux is not None:
            for aux_idx, col in enumerate(joint_aux_cols):
                ex = extra[col]
                ex[tt, :] = ex[tt - 1, :] + delta_aux[:, aux_idx]
                np.nan_to_num(ex[tt, :], copy=False, nan=0.0, posinf=SAFE_FEATURE_CLIP, neginf=-SAFE_FEATURE_CLIP)
                np.clip(ex[tt, :], -SAFE_FEATURE_CLIP, SAFE_FEATURE_CLIP, out=ex[tt, :])

    return {"t": t, "pred": y, "n": n}


def predict_event_recursive_weighted(ev, models, scaler_mean, scaler_std, anchor_params,
                                     start_t: int, y_lags: int, rain_lags: int,
                                     extra_cols, joint_aux_cols, extra_lags: int, static_feat: np.ndarray | None):
    score = compute_rain_event_score(ev)
    weights = anchor_norm_weights(score, anchor_params["centers"], anchor_params["bandwidth"])

    outs = [
        predict_event_recursive(ev, m, scaler_mean, scaler_std, start_t, y_lags, rain_lags, extra_cols, joint_aux_cols, extra_lags, static_feat)
        for m in models
    ]
    base = outs[0]
    pred = np.zeros_like(base["pred"], dtype=np.float32)
    for wk, out in zip(weights.tolist(), outs):
        pred += float(wk) * out["pred"].astype(np.float32, copy=False)
    return {"t": base["t"], "pred": pred, "n": base["n"]}


# =========================
# Metrics / Tables
# =========================

def evaluate_oof_lb_equivalent(events, model, scaler_mean, scaler_std, target,
                               start_t: int, y_lags: int, rain_lags: int,
                               extra_cols, joint_aux_cols, extra_lags: int, static_feat: np.ndarray | None):
    if OOF_LB_MAX_EVENTS > 0:
        eval_events = events[:OOF_LB_MAX_EVENTS]
    else:
        eval_events = events

    std = float(STD_DEV[target])
    event_scores = []
    node_scores = []

    total = len(eval_events)
    for j, ev in enumerate(eval_events, 1):
        out = predict_event_recursive(ev, model, scaler_mean, scaler_std, start_t, y_lags, rain_lags, extra_cols, joint_aux_cols, extra_lags, static_feat)
        n = out["n"]
        if n <= start_t:
            continue

        y_true = ev["y"][start_t:n, :]
        y_pred = out["pred"][start_t:n, :]
        err = y_pred - y_true

        rmse_node = np.sqrt(np.mean(err * err, axis=0))
        std_node = (rmse_node / std).astype(np.float32, copy=False)
        event_score = float(np.mean(std_node))

        event_scores.append(event_score)
        node_scores.append(std_node)

        if j % max(1, LOG_EVERY_EVENTS) == 0 or j == total:
            log(f"oof-lb eval progress: {j}/{total} event_std_rmse={event_score:.6f}")

    if not event_scores:
        return float("nan"), None

    node_mean = np.mean(np.stack(node_scores, axis=0), axis=0).astype(np.float32)
    return float(np.mean(event_scores)), node_mean


def evaluate_oof_lb_equivalent_weighted(events, models, scaler_mean, scaler_std, anchor_params, target,
                                        start_t: int, y_lags: int, rain_lags: int,
                                        extra_cols, joint_aux_cols, extra_lags: int, static_feat: np.ndarray | None):
    if OOF_LB_MAX_EVENTS > 0:
        eval_events = events[:OOF_LB_MAX_EVENTS]
    else:
        eval_events = events

    std = float(STD_DEV[target])
    event_scores = []
    node_scores = []

    total = len(eval_events)
    for j, ev in enumerate(eval_events, 1):
        out = predict_event_recursive_weighted(ev, models, scaler_mean, scaler_std, anchor_params, start_t, y_lags, rain_lags, extra_cols, joint_aux_cols, extra_lags, static_feat)
        n = out["n"]
        if n <= start_t:
            continue

        y_true = ev["y"][start_t:n, :]
        y_pred = out["pred"][start_t:n, :]
        err = y_pred - y_true

        rmse_node = np.sqrt(np.mean(err * err, axis=0))
        std_node = (rmse_node / std).astype(np.float32, copy=False)
        event_score = float(np.mean(std_node))

        event_scores.append(event_score)
        node_scores.append(std_node)

        if j % max(1, LOG_EVERY_EVENTS) == 0 or j == total:
            log(f"oof-lb eval progress: {j}/{total} event_std_rmse={event_score:.6f}")

    if not event_scores:
        return float("nan"), None

    node_mean = np.mean(np.stack(node_scores, axis=0), axis=0).astype(np.float32)
    return float(np.mean(event_scores)), node_mean


def build_prediction_df(model_id: int, node_type: int, events, node_ids, fold_states, extra_cols, joint_aux_cols, static_feat: np.ndarray | None):
    fold_parts = []

    for fold_idx, st in enumerate(fold_states):
        rows = []
        total = len(events)

        for j, ev in enumerate(events, 1):
            if st.get("anchor_models") is not None and st.get("anchor_params") is not None:
                out = predict_event_recursive_weighted(
                    ev=ev,
                    models=st["anchor_models"],
                    scaler_mean=st["scaler_mean"],
                    scaler_std=st["scaler_std"],
                    anchor_params=st["anchor_params"],
                    start_t=START_T,
                    y_lags=Y_LAGS,
                    rain_lags=RAIN_LAGS,
                    extra_cols=extra_cols,
                    joint_aux_cols=joint_aux_cols,
                    extra_lags=EXTRA_LAGS,
                    static_feat=static_feat,
                )
            else:
                out = predict_event_recursive(
                    ev=ev,
                    model=st["model"],
                    scaler_mean=st["scaler_mean"],
                    scaler_std=st["scaler_std"],
                    start_t=START_T,
                    y_lags=Y_LAGS,
                    rain_lags=RAIN_LAGS,
                    extra_cols=extra_cols,
                    joint_aux_cols=joint_aux_cols,
                    extra_lags=EXTRA_LAGS,
                    static_feat=static_feat,
                )

            n = out["n"]
            ts = out["t"][START_T:n]
            pred = out["pred"][START_T:n, :]
            if ts.size == 0:
                continue

            part = pd.DataFrame(
                {
                    "model_id": np.full((ts.size * pred.shape[1],), model_id, dtype=np.int8),
                    "event_id": np.full((ts.size * pred.shape[1],), ev["event_id"], dtype=np.int32),
                    "node_type": np.full((ts.size * pred.shape[1],), node_type, dtype=np.int8),
                    "node_id": np.tile(node_ids, ts.size),
                    "timestep": np.repeat(ts.astype(np.int32), pred.shape[1]),
                    "water_level": pred.reshape(-1).astype(np.float32, copy=False),
                }
            )
            rows.append(part)

            if j % max(1, LOG_EVERY_EVENTS) == 0 or j == total:
                log(f"predict fold={fold_idx+1} progress: {j}/{total} events")

        if rows:
            fold_parts.append(pd.concat(rows, ignore_index=True))
        gc.collect()

    if not fold_parts:
        return pd.DataFrame(columns=KEY_COLS + ["water_level"])

    pred = pd.concat(fold_parts, ignore_index=True)
    pred = pred.groupby(KEY_COLS, as_index=False)["water_level"].mean()
    return pred


def build_oof_prediction_df(model_id: int, node_type: int, train_events_all, node_ids, fold_states, extra_cols, joint_aux_cols, static_feat: np.ndarray | None):
    rows = []

    for st in fold_states:
        val_ids = st["val_ids"]
        va_events = []
        for eid in val_ids:
            ev = train_events_all[eid]
            va_events.append(
                {
                    "event_id": ev["event_id"],
                    "t": ev["t"],
                    "y": ev["y"].copy(),
                    "rain": ev["rain"],
                    "extra": {k: v.copy() for k, v in ev["extra"].items()},
                }
            )
        fill_missing(va_events, st["node_means"], st["extra_means"])

        total = len(va_events)
        for j, ev in enumerate(va_events, 1):
            if st.get("anchor_models") is not None and st.get("anchor_params") is not None:
                out = predict_event_recursive_weighted(
                    ev=ev,
                    models=st["anchor_models"],
                    scaler_mean=st["scaler_mean"],
                    scaler_std=st["scaler_std"],
                    anchor_params=st["anchor_params"],
                    start_t=START_T,
                    y_lags=Y_LAGS,
                    rain_lags=RAIN_LAGS,
                    extra_cols=extra_cols,
                    joint_aux_cols=joint_aux_cols,
                    extra_lags=EXTRA_LAGS,
                    static_feat=static_feat,
                )
            else:
                out = predict_event_recursive(
                    ev=ev,
                    model=st["model"],
                    scaler_mean=st["scaler_mean"],
                    scaler_std=st["scaler_std"],
                    start_t=START_T,
                    y_lags=Y_LAGS,
                    rain_lags=RAIN_LAGS,
                    extra_cols=extra_cols,
                    joint_aux_cols=joint_aux_cols,
                    extra_lags=EXTRA_LAGS,
                    static_feat=static_feat,
                )

            n = out["n"]
            ts = out["t"][START_T:n]
            pred = out["pred"][START_T:n, :]
            y_true = ev["y"][START_T:n, :]

            if ts.size == 0:
                continue

            m = min(ts.size, pred.shape[0], y_true.shape[0])
            if m <= 0:
                continue

            ts = ts[:m]
            pred = pred[:m, :]
            y_true = y_true[:m, :]

            part = pd.DataFrame(
                {
                    "model_id": np.full((m * pred.shape[1],), model_id, dtype=np.int8),
                    "event_id": np.full((m * pred.shape[1],), ev["event_id"], dtype=np.int32),
                    "node_type": np.full((m * pred.shape[1],), node_type, dtype=np.int8),
                    "node_id": np.tile(node_ids, m),
                    "timestep": np.repeat(ts.astype(np.int32), pred.shape[1]),
                    "water_level_true": y_true.reshape(-1).astype(np.float32, copy=False),
                    "water_level_pred": pred.reshape(-1).astype(np.float32, copy=False),
                }
            )
            rows.append(part)

            if j % max(1, LOG_EVERY_EVENTS) == 0 or j == total:
                log(f"oof table fold={st['fold_idx']+1} progress: {j}/{total} events")

        del va_events
        gc.collect()

    if not rows:
        return pd.DataFrame(columns=KEY_COLS + ["water_level_true", "water_level_pred"])

    oof = pd.concat(rows, ignore_index=True)
    oof = oof.sort_values(KEY_COLS).reset_index(drop=True)
    return oof


def build_submission(sample_path: Path, pred_df: pd.DataFrame, output_path: str):
    out_path = Path(output_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    sample = pd.read_csv(sample_path)
    sub = sample.copy()
    if "water_level" not in sub.columns:
        sub["water_level"] = np.nan

    if pred_df is None:
        pred_df = pd.DataFrame(columns=KEY_COLS + ["water_level"])

    if len(pred_df) > 0 and len(sample) == len(pred_df) and "water_level" in pred_df.columns:
        sub["water_level"] = pd.to_numeric(pred_df["water_level"], errors="coerce").to_numpy(dtype=np.float32, copy=False)
    elif len(pred_df) > 0 and "water_level" in pred_df.columns:
        if "timestep" in sub.columns:
            sub["__step_idx"] = sub["timestep"].astype(np.int32)
        else:
            sub["__step_idx"] = sub.groupby(["model_id", "event_id", "node_type", "node_id"]).cumcount()

        pred = pred_df.copy()
        if "timestep" in pred.columns:
            pred["__step_idx"] = pred["timestep"].astype(np.int32)
        else:
            pred["__step_idx"] = pred.groupby(["model_id", "event_id", "node_type", "node_id"]).cumcount()

        pred = pred[["model_id", "event_id", "node_type", "node_id", "__step_idx", "water_level"]]
        sub = sub.merge(pred, on=["model_id", "event_id", "node_type", "node_id", "__step_idx"], how="left", suffixes=("_base", "_pred"))

        if "water_level_pred" in sub.columns:
            sub["water_level"] = sub["water_level_pred"].where(
                sub["water_level_pred"].notna(),
                pd.to_numeric(sub.get("water_level_base"), errors="coerce"),
            )
            sub = sub.drop(columns=["water_level_base", "water_level_pred"], errors="ignore")

        sub = sub.drop(columns=["__step_idx"], errors="ignore")
    else:
        log("warning: pred_df empty or missing water_level; using sample water_level as-is")

    wl = pd.to_numeric(sub["water_level"], errors="coerce")
    missing = int(wl.isna().sum())
    if missing > 0:
        base_vals = pd.to_numeric(sample.get("water_level"), errors="coerce").to_numpy(dtype=np.float64, copy=False)
        fill_value = float(np.nanmean(base_vals)) if np.isfinite(base_vals).any() else 0.0
        wl = wl.fillna(fill_value)
        log(f"warning: filled missing submission rows with fallback value={fill_value:.6f} rows={missing}")

    sub["water_level"] = wl.astype(np.float32, copy=False)
    sub.to_csv(out_path, index=False)


# =========================
# Main
# =========================

def summarize_parallel_intervals(results):
    if not results:
        return {
            "count": 0,
            "max_concurrent": 0,
            "wall_span": 0.0,
            "sum_durations": 0.0,
        }
    intervals = []
    for r in results:
        start = float(r.get("worker_started_at", float("nan")))
        end = float(r.get("worker_finished_at", float("nan")))
        if not (np.isfinite(start) and np.isfinite(end) and end >= start):
            continue
        intervals.append((start, end))
    if not intervals:
        return {
            "count": 0,
            "max_concurrent": 0,
            "wall_span": 0.0,
            "sum_durations": 0.0,
        }
    points = []
    for start, end in intervals:
        points.append((start, 1))
        points.append((end, -1))
    points.sort(key=lambda x: (x[0], -x[1]))
    cur = 0
    max_cur = 0
    for _, delta in points:
        cur += delta
        if cur > max_cur:
            max_cur = cur
    starts = [s for s, _ in intervals]
    ends = [e for _, e in intervals]
    return {
        "count": len(intervals),
        "max_concurrent": int(max_cur),
        "wall_span": float(max(ends) - min(starts)),
        "sum_durations": float(sum(e - s for s, e in intervals)),
    }


def main():
    t0 = time.time()
    set_seed(SEED)

    model_id, node_type = TARGET

    # extra cols = feature extras + aux extras (aux also must exist in extra)
    requested_extra_cols = []
    for c in (EXTRA_COLS_REQ + JOINT_AUX_COLS_REQ):
        if c and c not in requested_extra_cols:
            requested_extra_cols.append(c)

    models_root = resolve_models_root(DATASET_ROOT)
    sample_path = resolve_sample_submission(SAMPLE_SUBMISSION_PATH, models_root)
    mdir = model_dir(models_root, model_id)

    train_ids = list_event_ids(mdir, "train")
    test_ids = list_event_ids(mdir, "test")
    if MAX_TRAIN_EVENTS > 0:
        train_ids = train_ids[:MAX_TRAIN_EVENTS]
    if MAX_TEST_EVENTS > 0:
        test_ids = test_ids[:MAX_TEST_EVENTS]

    if len(train_ids) < N_FOLDS:
        raise RuntimeError(f"not enough train events ({len(train_ids)}) for N_FOLDS={N_FOLDS}")

    rain_col = detect_rain_col(mdir, "train", train_ids[0])
    node_ids = load_node_ids(mdir, node_type)
    static_feat, static_cols_active = load_static_node_features(
        mdir=mdir,
        node_type=node_type,
        node_ids=node_ids,
        requested_cols=STATIC_COLS_REQ,
        standardize=STATIC_STANDARDIZE,
    )
    if not STATIC_FEATURES_ENABLE:
        static_feat = np.empty((len(node_ids), 0), dtype=np.float32)
        static_cols_active = []

    extra_cols_active = resolve_extra_cols(mdir, "train", train_ids[0], node_type, requested_extra_cols)
    joint_aux_cols_active = [c for c in JOINT_AUX_COLS_REQ if c in extra_cols_active]

    log(f"DEVICE(initial)={DEVICE} USE_AMP(initial)={USE_AMP} deferred_cuda_init=True")
    log(f"DATASET_ROOT={DATASET_ROOT}")
    log(f"SAMPLE_SUBMISSION_PATH={sample_path}")
    log(f"RUN_DIR={RUN_DIR}")
    log(f"OUTPUT_PATH={OUTPUT_PATH}")
    log(f"PRED_TABLE_PATH={PRED_TABLE_PATH}")
    log(f"BUNDLE_DIR={BUNDLE_DIR}")

    log(f"TARGET={TARGET} node_count={len(node_ids)}")
    log(f"train_events={len(train_ids)} test_events={len(test_ids)}")
    log(f"N_FOLDS={N_FOLDS} RUN_SINGLE_FOLD={RUN_SINGLE_FOLD} FOLD_INDEX={FOLD_INDEX} START_T={START_T}")
    log(f"Y_LAGS={Y_LAGS} RAIN_LAGS={RAIN_LAGS} STATIC_FEATURES_ENABLE={STATIC_FEATURES_ENABLE} STATIC_COLS_ACTIVE={static_cols_active} STATIC_DIM={int(static_feat.shape[1])}")
    log(f"EXTRA_COLS_ACTIVE={extra_cols_active} EXTRA_LAGS={EXTRA_LAGS}")
    log(f"JOINT_AUX_COLS_ACTIVE={joint_aux_cols_active} AUX_LOSS_WEIGHT={AUX_LOSS_WEIGHT}")
    log(f"RAIN3_ENABLED={RAIN3_ENABLED} Q=({RAIN_ANCHOR_QLOW},{RAIN_ANCHOR_QMID},{RAIN_ANCHOR_QHIGH}) BW_SCALE={RAIN_ANCHOR_BW_SCALE} MIN_BW={RAIN_ANCHOR_MIN_BW}")
    log(
        f"MODEL: D={GNN_D_MODEL} L={GNN_N_LAYERS} dropout={GNN_DROPOUT} node_emb={NODE_EMBED_DIM} "
        f"layer_type={GNN_LAYER_TYPE} attn_dropout={ATTN_DROPOUT} attn_out_proj={ATTN_USE_OUT_PROJ} "
        f"rain_gate_enable={RAIN_GATE_ENABLE} rain_gate_hidden={RAIN_GATE_HIDDEN} rain_gate_scale={RAIN_GATE_SCALE}"
    )
    log(f"ROLLOUT: enable={ROLLOUT_LOSS_ENABLE} steps={ROLLOUT_STEPS} weight={ROLLOUT_WEIGHT} batch_size={ROLLOUT_BATCH_SIZE} max_samples={ROLLOUT_MAX_SAMPLES}")
    log(f"TRAIN: EPOCHS={EPOCHS} BS={GRAPH_BATCH_SIZE} LR={LR} WD={WEIGHT_DECAY} PATIENCE={PATIENCE}")
    log(f"NUM_WORKERS={NUM_WORKERS} CPU_COUNT={CPU_COUNT}")
    log(f"PARALLEL_FOLDS: enable={PARALLEL_FOLDS_ENABLE} gpu_ids={PARALLEL_GPU_IDS} max_workers={PARALLEL_MAX_WORKERS} start_method={PARALLEL_START_METHOD}")

    edge_index_np = load_1d_edge_index(mdir, node_ids)
    log(f"graph edge_index loaded: edges={edge_index_np.shape[1]} (undirected expanded)")

    # preload
    log("Step 1: preload events")
    train_events_all = {}
    for j, eid in enumerate(train_ids, 1):
        train_events_all[eid] = load_event(mdir, "train", eid, node_type, node_ids, TARGET_COL, rain_col, extra_cols_active)
        if j % max(1, LOG_EVERY_EVENTS) == 0 or j == len(train_ids):
            log(f"preload train: {j}/{len(train_ids)}")

    test_events_all = {}
    for j, eid in enumerate(test_ids, 1):
        test_events_all[eid] = load_event(mdir, "test", eid, node_type, node_ids, TARGET_COL, rain_col, extra_cols_active)
        if j % max(1, LOG_EVERY_EVENTS) == 0 or j == len(test_ids):
            log(f"preload test: {j}/{len(test_ids)}")

    # folds
    folds = split_events_kfold(train_ids, N_FOLDS, EVENT_SPLIT_SEED + model_id * 1000 + node_type)
    fold_indices = [FOLD_INDEX] if RUN_SINGLE_FOLD else list(range(N_FOLDS))
    log(f"fold sizes={[len(f) for f in folds]} active_folds={[i+1 for i in fold_indices]}")

    fold_states = []
    sample_val_scores = []
    oof_lb_scores = []

    # train folds
    log("Step 2: fold training")
    fold_jobs = []
    for run_i, fold_idx in enumerate(fold_indices, 1):
        val_ids = folds[fold_idx]
        tr_ids = fold_train_events(folds, fold_idx)
        fold_jobs.append({
            "run_i": run_i,
            "total_runs": len(fold_indices),
            "fold_idx": fold_idx,
            "val_ids": list(val_ids),
            "tr_ids": list(tr_ids),
        })

    parallel_reasons = []
    gpu_count = detect_gpu_count_no_torch()
    if not PARALLEL_FOLDS_ENABLE:
        parallel_reasons.append('disabled_by_config')
    if PARALLEL_START_METHOD != "fork":
        parallel_reasons.append(f'start_method={PARALLEL_START_METHOD}')
    if gpu_count < 2:
        parallel_reasons.append(f'gpu_count={gpu_count}')
    if len(PARALLEL_GPU_IDS) < 2:
        parallel_reasons.append(f'gpu_ids={PARALLEL_GPU_IDS}')
    if len(fold_jobs) < 2:
        parallel_reasons.append(f'fold_jobs={len(fold_jobs)}')

    use_parallel_folds = len(parallel_reasons) == 0
    log(f"parallel fold gate: use_parallel={use_parallel_folds} reasons={parallel_reasons if parallel_reasons else ['ok']} fold_jobs={len(fold_jobs)} gpu_count={gpu_count}")

    if use_parallel_folds:
        init_parallel_context(
            train_events_all=train_events_all,
            node_ids=node_ids,
            edge_index_np=edge_index_np,
            extra_cols_active=extra_cols_active,
            joint_aux_cols_active=joint_aux_cols_active,
            static_feat=static_feat,
            static_cols_active=static_cols_active,
        )
        max_workers = len(PARALLEL_GPU_IDS)
        if PARALLEL_MAX_WORKERS > 0:
            max_workers = min(max_workers, PARALLEL_MAX_WORKERS)
        max_workers = min(max_workers, len(fold_jobs))
        log(f"Step 2: fold training (parallel) workers={max_workers} start_method={PARALLEL_START_METHOD}")
        fold_results = []
        try:
            ctx = mp.get_context(PARALLEL_START_METHOD)
            with ProcessPoolExecutor(max_workers=max_workers, mp_context=ctx) as ex:
                futures = []
                for job_pos, job in enumerate(fold_jobs):
                    job = dict(job)
                    job["gpu_id"] = PARALLEL_GPU_IDS[job_pos % len(PARALLEL_GPU_IDS)]
                    log(f"parallel submit: fold={job['fold_idx']+1} gpu={job['gpu_id']}")
                    futures.append(ex.submit(run_fold_training_job, job))
                for fut in as_completed(futures):
                    result = fut.result()
                    fold_results.append(result)
                    log(
                        f"fold worker done: fold={result['fold_idx']+1} gpu={result.get('gpu_id')} "
                        f"sample_std={result['sample_rmse']/float(STD_DEV[TARGET]):.6f} oof_lb_std={result['oof_lb_score']:.6f}"
                    )
        except Exception as exc:
            log(f"parallel fold training failed; fallback to sequential: {exc}")
            fold_results = []
            use_parallel_folds = False

        if use_parallel_folds:
            set_runtime_device(PARALLEL_GPU_IDS[0])
            log(f"parent runtime device activated after parallel training: DEVICE={DEVICE} USE_AMP={USE_AMP} gpu_id={PARALLEL_GPU_IDS[0]}")
            timing = summarize_parallel_intervals(fold_results)
            log(f"parallel timing summary: folds={timing['count']} max_concurrent={timing['max_concurrent']} wall_span={timing['wall_span']:.1f}s sum_worker_durations={timing['sum_durations']:.1f}s")
            fold_results.sort(key=lambda x: x["fold_idx"])
            for result in fold_results:
                sample_val_scores.append(float(result["sample_rmse"]))
                oof_lb_scores.append(float(result["oof_lb_score"]))
                fold_states.append(restore_fold_state_from_result(result, edge_index_np))

    if not use_parallel_folds:
        if gpu_count > 0:
            set_runtime_device(PARALLEL_GPU_IDS[0] if PARALLEL_GPU_IDS else 0)
        else:
            set_runtime_device(None)
        log(f"parent runtime device activated for sequential training: DEVICE={DEVICE} USE_AMP={USE_AMP}")
        for run_i, fold_idx in enumerate(fold_indices, 1):
            val_ids = folds[fold_idx]
            tr_ids = fold_train_events(folds, fold_idx)

            log(f"fold {fold_idx+1}/{N_FOLDS} (run {run_i}/{len(fold_indices)}): train_events={len(tr_ids)} val_events={len(val_ids)}")

            tr_events = [train_events_all[eid] for eid in tr_ids]
            va_events = [train_events_all[eid] for eid in val_ids]

            means = compute_node_means_from_events(tr_events)
            extra_means = compute_extra_means_from_events(tr_events, extra_cols_active)

            # per-fold copies
            tr_events = [
                {"event_id": ev["event_id"], "t": ev["t"], "y": ev["y"].copy(), "rain": ev["rain"], "extra": {k: v.copy() for k, v in ev["extra"].items()}}
                for ev in tr_events
            ]
            va_events = [
                {"event_id": ev["event_id"], "t": ev["t"], "y": ev["y"].copy(), "rain": ev["rain"], "extra": {k: v.copy() for k, v in ev["extra"].items()}}
                for ev in va_events
            ]

            fill_missing(tr_events, means, extra_means)
            fill_missing(va_events, means, extra_means)

            seed_base = SEED + fold_idx * 100

            if RAIN3_ENABLED:
                x_train, y_train, y_train_aux, rows_pos_train, rows_kept_train, graph_event_ids_train = build_samples(
                    tr_events, START_T, Y_LAGS, RAIN_LAGS, extra_cols_active, joint_aux_cols_active, EXTRA_LAGS, static_feat, MAX_TRAIN_ROWS, seed_base,
                    return_graph_event_ids=True,
                )
            else:
                x_train, y_train, y_train_aux, rows_pos_train, rows_kept_train = build_samples(
                    tr_events, START_T, Y_LAGS, RAIN_LAGS, extra_cols_active, joint_aux_cols_active, EXTRA_LAGS, static_feat, MAX_TRAIN_ROWS, seed_base,
                    return_graph_event_ids=False,
                )
                graph_event_ids_train = None

            x_val, y_val, y_val_aux, rows_pos_val, rows_kept_val = build_samples(
                va_events, START_T, Y_LAGS, RAIN_LAGS, extra_cols_active, joint_aux_cols_active, EXTRA_LAGS, static_feat, MAX_VAL_ROWS, seed_base + 1,
                return_graph_event_ids=False,
            )

            rollout_train_data, rollout_pos_train, rollout_kept_train = build_rollout_samples(
                tr_events,
                start_t=START_T,
                rollout_steps=ROLLOUT_STEPS,
                y_lags=Y_LAGS,
                rain_lags=RAIN_LAGS,
                extra_cols=extra_cols_active,
                joint_aux_cols=joint_aux_cols_active,
                extra_lags=EXTRA_LAGS,
                max_samples=ROLLOUT_MAX_SAMPLES,
                seed=seed_base + 11,
            )

            log(f"fold {fold_idx+1}: train_samples={rows_kept_train}/{rows_pos_train} val_samples={rows_kept_val}/{rows_pos_val} feat_dim={x_train.shape[1]} rollout_samples={rollout_kept_train}/{rollout_pos_train}")

            scaler_mean, scaler_std = fit_scaler(x_train)
            x_train = apply_scaler(x_train, scaler_mean, scaler_std)
            x_val = apply_scaler(x_val, scaler_mean, scaler_std)

            anchor_models = None
            anchor_params = None
            selected_checkpoint = "val"

            if RAIN3_ENABLED:
                anchor_params = build_rain_anchor_params(tr_events)
                graph_sample_w = build_anchor_graph_weights(
                    graph_event_ids=graph_event_ids_train,
                    score_by_event=anchor_params["score_by_event"],
                    centers=anchor_params["centers"],
                    bandwidth=anchor_params["bandwidth"],
                )
                log(
                    "rain3 anchors: "
                    f"centers=[{anchor_params['centers'][0]:.4f}, {anchor_params['centers'][1]:.4f}, {anchor_params['centers'][2]:.4f}] "
                    f"bandwidth={anchor_params['bandwidth']:.4f}"
                )

                anchor_models = []
                anchor_best_rmses = []

                for anchor_idx, anchor_name in enumerate(("dry", "mid", "wet")):
                    w_anchor = graph_sample_w[:, anchor_idx].astype(np.float32, copy=False)
                    use_w = w_anchor if float(np.sum(w_anchor)) > 0.0 else None
                    if use_w is None:
                        log(f"anchor fallback: fold={fold_idx+1} name={anchor_name} (no positive weights)")

                    log(f"anchor train: fold={fold_idx+1}/{N_FOLDS} name={anchor_name} graph_samples={x_train.shape[0] // max(1, len(node_ids))}")

                    fit_result_k = train_one_fold(
                        x_train, y_train, y_train_aux,
                        x_val, y_val, y_val_aux,
                        fold_idx,
                        n_nodes=len(node_ids),
                        edge_index_np=edge_index_np,
                        extra_cols=extra_cols_active,
                        joint_aux_cols=joint_aux_cols_active,
                        extra_lags=EXTRA_LAGS,
                        sample_weight_train_graph=use_w,
                        epoch_oof_eval_every=0,
                        epoch_oof_eval_fn=None,
                        aux_loss_weight=AUX_LOSS_WEIGHT,
                        rollout_train_data=rollout_train_data,
                        scaler_mean=scaler_mean,
                        scaler_std=scaler_std,
                        static_feat=static_feat,
                    )
                    model_k = fit_result_k["model"]
                    best_rmse_k = float(fit_result_k["best_val_rmse"])
                    anchor_models.append(model_k)
                    anchor_best_rmses.append(best_rmse_k)
                    log(f"anchor done: fold={fold_idx+1}/{N_FOLDS} name={anchor_name} sample_std={best_rmse_k/float(STD_DEV[TARGET]):.6f}")

                # representative model (mid)
                model = anchor_models[1]
                best_rmse = float(np.mean(anchor_best_rmses))
                sample_val_scores.append(best_rmse)

                del graph_sample_w
            else:
                # optional epoch oof eval
                epoch_eval_events = va_events[:EPOCH_OOF_MAX_EVENTS] if EPOCH_OOF_MAX_EVENTS > 0 else va_events

                if OOF_LB_EVAL and EPOCH_OOF_EVAL_EVERY > 0 and len(epoch_eval_events) > 0:
                    def _epoch_oof_eval_fn(model_cur):
                        score, _ = evaluate_oof_lb_equivalent(
                            events=epoch_eval_events,
                            model=model_cur,
                            scaler_mean=scaler_mean,
                            scaler_std=scaler_std,
                            target=TARGET,
                            start_t=START_T,
                            y_lags=Y_LAGS,
                            rain_lags=RAIN_LAGS,
                            extra_cols=extra_cols_active,
                            joint_aux_cols=joint_aux_cols_active,
                            extra_lags=EXTRA_LAGS,
                            static_feat=static_feat,
                        )
                        return float(score)
                else:
                    _epoch_oof_eval_fn = None

                fit_result = train_one_fold(
                    x_train, y_train, y_train_aux,
                    x_val, y_val, y_val_aux,
                    fold_idx,
                    n_nodes=len(node_ids),
                    edge_index_np=edge_index_np,
                    extra_cols=extra_cols_active,
                    joint_aux_cols=joint_aux_cols_active,
                    extra_lags=EXTRA_LAGS,
                    sample_weight_train_graph=None,
                    epoch_oof_eval_every=EPOCH_OOF_EVAL_EVERY,
                    epoch_oof_eval_fn=_epoch_oof_eval_fn,
                    aux_loss_weight=AUX_LOSS_WEIGHT,
                    rollout_train_data=rollout_train_data,
                    scaler_mean=scaler_mean,
                    scaler_std=scaler_std,
                    static_feat=static_feat,
                )
                model = fit_result["model"]
                best_rmse = float(fit_result["best_val_rmse"])
                best_val_state = fit_result["best_val_state"]
                best_epoch_oof_std = float(fit_result["best_epoch_oof_std"])
                best_oof_state = fit_result["best_oof_state"]
                sample_val_scores.append(best_rmse)

            # oof-lb eval / checkpoint selection
            if OOF_LB_EVAL:
                if RAIN3_ENABLED:
                    oof_lb_score, _ = evaluate_oof_lb_equivalent_weighted(
                        events=va_events,
                        models=anchor_models,
                        scaler_mean=scaler_mean,
                        scaler_std=scaler_std,
                        anchor_params=anchor_params,
                        target=TARGET,
                        start_t=START_T,
                        y_lags=Y_LAGS,
                        rain_lags=RAIN_LAGS,
                        extra_cols=extra_cols_active,
                        joint_aux_cols=joint_aux_cols_active,
                        extra_lags=EXTRA_LAGS,
                        static_feat=static_feat,
                    )
                else:
                    val_checkpoint_oof = float("nan")
                    epoch_checkpoint_oof = float("nan")

                    model.load_state_dict(best_val_state)
                    val_checkpoint_oof, _ = evaluate_oof_lb_equivalent(
                        events=va_events,
                        model=model,
                        scaler_mean=scaler_mean,
                        scaler_std=scaler_std,
                        target=TARGET,
                        start_t=START_T,
                        y_lags=Y_LAGS,
                        rain_lags=RAIN_LAGS,
                        extra_cols=extra_cols_active,
                        joint_aux_cols=joint_aux_cols_active,
                        extra_lags=EXTRA_LAGS,
                        static_feat=static_feat,
                    )

                    selected_checkpoint = "val"
                    selected_state = best_val_state
                    oof_lb_score = float(val_checkpoint_oof)

                    if best_oof_state is not None:
                        model.load_state_dict(best_oof_state)
                        epoch_checkpoint_oof, _ = evaluate_oof_lb_equivalent(
                            events=va_events,
                            model=model,
                            scaler_mean=scaler_mean,
                            scaler_std=scaler_std,
                            target=TARGET,
                            start_t=START_T,
                            y_lags=Y_LAGS,
                            rain_lags=RAIN_LAGS,
                            extra_cols=extra_cols_active,
                            joint_aux_cols=joint_aux_cols_active,
                            extra_lags=EXTRA_LAGS,
                            static_feat=static_feat,
                        )
                        if np.isfinite(epoch_checkpoint_oof) and (not np.isfinite(oof_lb_score) or epoch_checkpoint_oof < oof_lb_score):
                            selected_checkpoint = "epoch_oof"
                            selected_state = best_oof_state
                            oof_lb_score = float(epoch_checkpoint_oof)

                    model.load_state_dict(selected_state)
                    tracked_epoch_oof_std = best_epoch_oof_std if np.isfinite(best_epoch_oof_std) else float("nan")
                    log(
                        f"fold {fold_idx+1} checkpoint_select: val_best_std={val_checkpoint_oof:.6f} "
                        f"epoch_oof_best_std={epoch_checkpoint_oof:.6f} "
                        f"epoch_oof_tracked={tracked_epoch_oof_std:.6f} chosen={selected_checkpoint}"
                    )
            else:
                oof_lb_score = float("nan")

            oof_lb_scores.append(oof_lb_score)

            log(
                f"fold {fold_idx+1} summary: sample_std={best_rmse/float(STD_DEV[TARGET]):.6f} "
                f"oof_lb_std={oof_lb_score:.6f}"
            )

            fold_states.append(
                {
                    "fold_idx": fold_idx,
                    "val_ids": list(val_ids),
                    "model": model,
                    "scaler_mean": scaler_mean,
                    "scaler_std": scaler_std,
                    "selected_checkpoint": selected_checkpoint if not RAIN3_ENABLED else "val",
                    "node_means": means,
                    "extra_means": extra_means,
                    "anchor_models": anchor_models,
                    "anchor_params": anchor_params,
                    "static_cols": list(static_cols_active),
                    "static_dim": int(static_feat.shape[1]),
                }
            )

            del x_train, y_train, y_train_aux, x_val, y_val, y_val_aux, rollout_train_data, tr_events, va_events
            if graph_event_ids_train is not None:
                del graph_event_ids_train
            gc.collect()

    mean_sample_rmse = float(np.mean(sample_val_scores)) if sample_val_scores else float("nan")
    mean_sample_std = mean_sample_rmse / float(STD_DEV[TARGET]) if np.isfinite(mean_sample_rmse) else float("nan")
    valid_oof = [x for x in oof_lb_scores if np.isfinite(x)]
    mean_oof_lb = float(np.mean(valid_oof)) if valid_oof else float("nan")

    log(f"CV done: sample_rmse={mean_sample_rmse:.6f} sample_std={mean_sample_std:.6f} oof_lb_std={mean_oof_lb:.6f} executed_folds={[i+1 for i in fold_indices]}")

    # predict test (fold avg)
    log("Step 3: recursive prediction (fold average)")

    avg_means = np.mean(np.stack([st["node_means"] for st in fold_states], axis=0), axis=0).astype(np.float32)
    avg_extra_means = {
        col: np.mean(np.stack([st["extra_means"][col] for st in fold_states], axis=0), axis=0).astype(np.float32)
        for col in extra_cols_active
    }

    test_events = []
    for eid in test_ids:
        ev = test_events_all[eid]
        test_events.append(
            {"event_id": ev["event_id"], "t": ev["t"], "y": ev["y"].copy(), "rain": ev["rain"], "extra": {k: v.copy() for k, v in ev["extra"].items()}}
        )
    fill_missing(test_events, avg_means, avg_extra_means)

    pred_df = build_prediction_df(model_id, node_type, test_events, node_ids, fold_states, extra_cols_active, joint_aux_cols_active, static_feat)
    pred_df = pred_df.sort_values(KEY_COLS).reset_index(drop=True)

    log(f"prediction rows={len(pred_df)}")

    # save prediction table
    pred_df.to_csv(Path(PRED_TABLE_PATH), index=False)
    log(f"prediction table saved: {PRED_TABLE_PATH}")

    # submission
    build_submission(sample_path=sample_path, pred_df=pred_df, output_path=OUTPUT_PATH)
    log(f"submission saved: {OUTPUT_PATH}")

    # bundle
    log("Step 4: save bundle outputs")
    oof_df = build_oof_prediction_df(model_id, node_type, train_events_all, node_ids, fold_states, extra_cols_active, joint_aux_cols_active, static_feat)

    test_df = pred_df[KEY_COLS + ["water_level"]].copy()
    test_df = test_df.rename(columns={"water_level": "water_level_pred"})
    try:
        ss = pd.read_csv(sample_path, usecols=["row_id"] + KEY_COLS)
        test_df = test_df.merge(ss, on=KEY_COLS, how="left")
    except Exception:
        pass

    oof_path = save_table_auto(oof_df, BUNDLE_DIR / "oof_predictions")
    test_path = save_table_auto(test_df, BUNDLE_DIR / "test_predictions")

    submission_copy_path = BUNDLE_DIR / "submission.csv"
    pd.read_csv(OUTPUT_PATH).to_csv(submission_copy_path, index=False)

    cv_summary = pd.DataFrame(
        [
            {
                "model_id": model_id,
                "node_type": node_type,
                "target_col": TARGET_COL,
                "n_folds": N_FOLDS,
                "executed_folds": ",".join(str(i + 1) for i in fold_indices),
                "event_split_seed": EVENT_SPLIT_SEED,
                "sample_rmse": mean_sample_rmse,
                "sample_std": mean_sample_std,
                "oof_lb_std": mean_oof_lb,
                "oof_rows": len(oof_df),
                "test_rows": len(test_df),
                "rain3_enabled": int(RAIN3_ENABLED),
                "static_features_enable": int(STATIC_FEATURES_ENABLE),
                "static_cols": ",".join(static_cols_active),
                "static_dim": int(static_feat.shape[1]),
                "joint_aux_cols": ",".join(joint_aux_cols_active),
                "aux_loss_weight": float(AUX_LOSS_WEIGHT),
                "layer_type": GNN_LAYER_TYPE,
                "attn_dropout": float(ATTN_DROPOUT),
                "rollout_loss_enable": int(ROLLOUT_LOSS_ENABLE),
                "rollout_steps": int(ROLLOUT_STEPS),
                "rollout_weight": float(ROLLOUT_WEIGHT),
                "rain_gate_enable": int(RAIN_GATE_ENABLE),
                "rain_gate_hidden": int(RAIN_GATE_HIDDEN),
                "rain_gate_scale": float(RAIN_GATE_SCALE),
            }
        ]
    )
    cv_summary_path = BUNDLE_DIR / "cv_summary.csv"
    cv_summary.to_csv(cv_summary_path, index=False)

    manifest = {
        "schema_version": "ufb_ensemble_v1",
        "bundle_name": BUNDLE_NAME,
        "source_notebook": "physics-informed-gnn-static-attention-rollout-v2-raingate",
        "model_family": "physics_informed_gnn",
        "rain_mode": "rain3" if RAIN3_ENABLED else "single",
        "rain_anchor_q": [float(RAIN_ANCHOR_QLOW), float(RAIN_ANCHOR_QMID), float(RAIN_ANCHOR_QHIGH)],
        "rain_anchor_bw_scale": float(RAIN_ANCHOR_BW_SCALE),
        "rain_anchor_min_bw": float(RAIN_ANCHOR_MIN_BW),
        "model_id": model_id,
        "node_type": node_type,
        "target_col": TARGET_COL,
        "joint_aux_cols": list(joint_aux_cols_active),
        "static_features_enable": bool(STATIC_FEATURES_ENABLE),
        "static_cols": list(static_cols_active),
        "static_dim": int(static_feat.shape[1]),
        "aux_loss_weight": float(AUX_LOSS_WEIGHT),
        "layer_type": GNN_LAYER_TYPE,
        "attn_dropout": float(ATTN_DROPOUT),
        "rollout_loss_enable": bool(ROLLOUT_LOSS_ENABLE),
        "rollout_steps": int(ROLLOUT_STEPS),
        "rollout_weight": float(ROLLOUT_WEIGHT),
        "rain_gate_enable": bool(RAIN_GATE_ENABLE),
        "rain_gate_hidden": int(RAIN_GATE_HIDDEN),
        "rain_gate_scale": float(RAIN_GATE_SCALE),
        "n_folds": N_FOLDS,
        "executed_folds_count": len(fold_indices),
        "executed_folds": [int(i + 1) for i in fold_indices],
        "event_split_seed": EVENT_SPLIT_SEED,
        "oof_predictions_path": oof_path.name,
        "test_predictions_path": test_path.name,
        "submission_path": submission_copy_path.name,
        "cv_summary_path": cv_summary_path.name,
        "created_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    manifest_path = BUNDLE_DIR / "bundle_manifest.json"
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))

    log(f"bundle saved: {BUNDLE_DIR}")
    log(f"bundle files: {oof_path.name}, {test_path.name}, {manifest_path.name}, {cv_summary_path.name}, {submission_copy_path.name}")
    log(f"done elapsed={time.time()-t0:.1f}s")


if __name__ == "__main__":
    main()


[04:54:51] DEVICE(initial)=cpu USE_AMP(initial)=False deferred_cuda_init=True
[04:54:51] DATASET_ROOT=/kaggle/input/datasets/mtmrs1/urban-flood-bench-re
[04:54:51] SAMPLE_SUBMISSION_PATH=/kaggle/input/competitions/urban-flood-modelling/sample_submission.csv
[04:54:51] RUN_DIR=/kaggle/working/run_20260314_045451
[04:54:51] OUTPUT_PATH=/kaggle/working/run_20260314_045451/submission.csv
[04:54:51] PRED_TABLE_PATH=/kaggle/working/run_20260314_045451/pred_physgnn_m2_1d_static_attn_rollout_v2_raingate.csv
[04:54:51] BUNDLE_DIR=/kaggle/working/run_20260314_045451/bundle_physics_informed_gnn_m2_1d_static_attn_rollout_v2_raingate
[04:54:51] TARGET=(2, 1) node_count=198
[04:54:51] train_events=69 test_events=30
[04:54:51] N_FOLDS=5 RUN_SINGLE_FOLD=False FOLD_INDEX=0 START_T=10
[04:54:51] Y_LAGS=10 RAIN_LAGS=80 STATIC_FEATURES_ENABLE=True STATIC_COLS_ACTIVE=['position_x', 'position_y', 'depth', 'invert_elevation', 'surface_elevation', 'base_area'] STATIC_DIM=6
[04:54:51] EXTRA_COLS_ACTIVE=['inlet